# Objetivo 2 - v5: evaluacion 100 % auto-supervisada (backbone congelado)

**Proyecto de Grado UNAB - generado por `scripts/build_notebook_v5.py`. No editar a mano.**

## Que cambia respecto a v4, y por que

En v4 el "SSL" arrancaba de pesos de ImageNet (celda 7: `resnet50(weights=IMAGENET1K_V2)`)
y despues el fine-tune descongelaba todo el backbone usando etiquetas. Es un pipeline
valido, pero **no** es auto-supervisado puro: hubo 1.28 millones de etiquetas humanas en
el camino.

v5 cambia la regla del juego: **ninguna etiqueta toca el extractor de caracteristicas.**
El backbone se congela y solo se ajusta un clasificador diminuto encima. Asi la pregunta
"cuanto sabe el backbone" se responde sin contaminacion.

| | v4 | v5 |
|---|---|---|
| Init del SSL | ImageNet supervisado | DINOv2 (entrenado **sin etiquetas**) |
| Backbone en evaluacion | se re-entrena (15 epocas) | **congelado** |
| Probes | linear | **kNN + prototipos + linear** |
| Regimenes de etiquetas | 10/20/50/100 % | **1/5**/10/20/50/100 % |
| Rutas de datos | `/kaggle/input` fijo | 3 backends (Kaggle / kagglehub / local) |
| Costo de repetir | re-entrenar todo | features cacheadas -> segundos |

## Como leerlo si no sabes nada de esto

- **Backbone**: la red que convierte una imagen en una lista de numeros ("features").
- **Congelado**: no se modifica. Solo se lee.
- **Probe**: clasificador minusculo encima de esas features. Mide la calidad de las
  features, no la del entrenamiento.
- **kNN**: clasifica una imagen mirando a que imagenes de entrenamiento se parece mas.
  **No entrena nada**: es la medida mas limpia posible.
- **Regimen de etiquetas**: cuantas etiquetas le damos al probe. Si el SSL sirve, con el
  1 % ya deberia andar bien.

## Como correrlo

1. Settings -> Accelerator **GPU T4 x2**; Internet **ON** (la primera vez, para bajar
   los pesos de DINOv2).
2. Add Input: `Skin Cancer MNIST: HAM10000` (kmader). Opcional: ISIC 2019 / 2020.
3. `SMOKE = True` -> Run All (~5 min). Valida que el pipeline entero corre.
4. `SMOKE = False` -> **Save & Run All (Commit)** para la corrida real.

El notebook nunca entrena el backbone, asi que **no necesita las GPUs de la universidad**:
ViT-S/14 congelado entra en una T4 de sobra. El pre-entrenamiento continuo sobre el pool
dermatoscopico (la etapa que si necesita el cluster) es el paso siguiente, y consumira
exactamente estas mismas features y probes.

In [ ]:
import gc, json, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageFile
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True  # una imagen truncada NO debe tumbar el run
T_START = time.time()

def elapsed():
    return f"{(time.time() - T_START) / 60:.1f} min"

# ----------------------------------------------------------------------------
# PERILLAS. Lo unico que se toca a mano.
# ----------------------------------------------------------------------------
SMOKE        = True          # True = validacion rapida del pipeline (numeros NO validos)
DATA_BACKEND = "auto"        # "auto" | "kaggle-input" | "kagglehub" | "local"
LOCAL_ROOT   = None          # ruta de datos ya bajados (para el cluster)
BASE_RES     = 224           # se ajusta por backbone (ViT patch 14 vs 16)
BATCH        = 64
NUM_WORKERS  = 4             # Kaggle da ~4 vCPU con GPU
MAX_HOURS    = 10.5          # guarda de tiempo (Kaggle corta a las 12 h)

# Backbones a comparar. Ninguno se entrena: solo se leen sus features.
#   ssl_*  -> pre-entrenado SIN etiquetas  (la hipotesis)
#   sup_*  -> pre-entrenado CON etiquetas  (el control)
BACKBONES = [
    "ssl_dinov2_vits14",   # DINOv2 ViT-S/14 - barato, cabe en T4 de sobra
    "sup_vit_b16",         # ViT-B/16 ImageNet supervisado - CONTROL de arquitectura
    "sup_resnet50",        # ResNet-50 ImageNet supervisado - linea base de v4
    "sup_alexnet",         # AlexNet 2012 - piso historico de la escalera
    "sup_vgg19",           # VGG19 2014 - profundidad bruta (sin BN, como el paper)
    "sup_effnet_b0",       # EfficientNet-B0 2019 - eficiencia (5.3M params)
]
# Cinco probes. Los "_balanced" reponderan por frecuencia de clase: se reportan AL
# LADO de los normales, no en su lugar, porque son estimadores distintos y cual
# conviene es una pregunta empirica que la corrida responde.
PROBES    = ["knn", "knn_balanced", "prototype", "logreg", "logreg_balanced"]
FRACTIONS = (0.01, 0.05, 0.10, 0.20, 0.50, 1.00)
SEEDS     = [42, 1234, 2026]

if SMOKE:
    BACKBONES = ["ssl_dinov2_vits14", "sup_resnet50"]
    PROBES    = ["knn", "knn_balanced", "logreg"]
    FRACTIONS = (0.05, 1.00)
    SEEDS     = [42]
    print("*** MODO SMOKE: los numeros NO son validos como resultado. "
          "Solo prueba que el pipeline corre entero. ***")

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("./out")
WORK.mkdir(parents=True, exist_ok=True)
CACHE_DIR = WORK / "features"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = device.type == "cuda"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

print("torch:", torch.__version__, "| device:", device, "| GPUs:", torch.cuda.device_count())
print(f"backbones={BACKBONES}\nprobes={PROBES}\nfracciones={FRACTIONS} | semillas={SEEDS}")

## 0. Nucleo reutilizable (generado, no editar aqui)

Las funciones de abajo son **copia exacta** de `src/dermssl/` en este repositorio, donde
estan cubiertas por `tests/` (83 pruebas que corren sin GPU, sin Kaggle y sin internet).

Se pegan aca porque una sesion de Kaggle no puede importar codigo del repositorio. El
notebook se genera con `python scripts/build_notebook_v5.py`, de modo que **el codigo que
corre aca y el codigo probado son el mismo codigo**. Para cambiar algo: se edita el modulo,
se corre `pytest`, y se regenera el notebook.

Que hay aca:

| Modulo | Responsabilidad |
|---|---|
| `patch_guard` | evita la trampa de resolucion vs tamano de parche (256 no es multiplo de 14) |
| `data_resolver` | localiza los datasets en Kaggle, por kagglehub o en disco local |
| `label_regimes` | muestreo de regimenes de pocas etiquetas, anidados y sin clases vacias |
| `metrics` | macro-F1, AUC, ECE, bootstrap - implementados en numpy y verificados a mano |
| `probes` | kNN, prototipos y linear probe sobre features congeladas |
| `feature_cache` | cache en disco de las features (resume + rejilla de experimentos barata) |

In [ ]:
# Inlined core of src/dermssl/ -- single source of truth, covered by tests/.
# Regenerate with: python scripts/build_notebook_v5.py
# NOTE: no 'from __future__ import annotations' on purpose - see module_source().

# ============================================================================
# patch_guard.py  --  inlined from src/dermssl/patch_guard.py (do not edit here)
# ============================================================================

"""Resolution guardrails for patch-based backbones (ViT).

Why this module exists
----------------------
A Vision Transformer cuts the image into a grid of square *patches*. DINOv2 uses
14x14 patches, so its input side must be an exact multiple of 14. A ResNet has
no such constraint, which is why v4 never needed this check.

The trap: 224 = 16 x 14 works, but 256 is **not** a multiple of 14 (256 / 14 =
18.28...). The ISIC 2020 dataset we use is distributed at 256x256. Feeding 256
straight into DINOv2 either raises a shape error deep inside attention or, worse,
silently interpolates the positional embeddings and quietly degrades the
features. Both outcomes waste a cluster run.

This module makes the constraint explicit and loud, before any GPU time is spent.
"""


KNOWN_PATCH_SIZES = {
    "dinov2_vits14": 14,
    "dinov2_vitb14": 14,
    "dinov2_vitl14": 14,
    "vit_base_patch16_224": 16,
    "vit_small_patch16_224": 16,
    "resnet50": None,  # convolutional: no patch constraint
    "alexnet": None,
    "vgg19": None,  # trailing digits must not trip the name heuristic
    "efficientnet_b0": None,
}


class PatchSizeError(ValueError):
    """Raised when an input resolution is incompatible with a patch size."""


def patch_size_for(backbone: str) -> int | None:
    """Patch size for a known backbone, or None when it has no constraint.

    Unknown names fall back to a name-based heuristic (``...14`` -> 14,
    ``...16`` -> 16) so a new checkpoint does not silently skip the check.
    """
    if backbone in KNOWN_PATCH_SIZES:
        return KNOWN_PATCH_SIZES[backbone]
    for suffix, size in (("14", 14), ("16", 16), ("32", 32), ("8", 8)):
        if backbone.endswith(suffix) or f"patch{suffix}" in backbone:
            return size
    return None


def is_compatible(resolution: int, patch: int | None) -> bool:
    """True when ``resolution`` divides evenly into ``patch``-sized tiles."""
    if patch is None:
        return True
    return resolution > 0 and resolution % patch == 0


def snap_to_patch(resolution: int, patch: int | None, mode: str = "down") -> int:
    """Round ``resolution`` to the nearest valid multiple of ``patch``.

    ``mode='down'`` never increases memory use, which is the safe default on a
    16 GB T4. ``mode='up'`` keeps a little more of the image. ``mode='nearest'``
    minimises the change.
    """
    if patch is None:
        return resolution
    if resolution <= 0:
        raise PatchSizeError(f"resolution must be positive, got {resolution}")

    lower = (resolution // patch) * patch
    upper = lower + patch if lower < resolution else lower
    if lower == 0:
        lower = patch

    if mode == "down":
        return lower
    if mode == "up":
        return upper
    if mode == "nearest":
        return lower if (resolution - lower) <= (upper - resolution) else upper
    raise ValueError(f"unknown mode {mode!r}; expected 'down', 'up' or 'nearest'")


def assert_compatible(resolution: int, backbone: str) -> int:
    """Validate ``resolution`` for ``backbone``; return it unchanged when valid.

    Raises :class:`PatchSizeError` with the two nearest valid resolutions so the
    fix is obvious from the message alone.
    """
    patch = patch_size_for(backbone)
    if is_compatible(resolution, patch):
        return resolution
    down = snap_to_patch(resolution, patch, "down")
    up = snap_to_patch(resolution, patch, "up")
    raise PatchSizeError(
        f"resolution {resolution} is not a multiple of patch size {patch} "
        f"(required by '{backbone}'). Use {down} or {up} instead."
    )


def resolve_resolution(requested: int, backbone: str, mode: str = "down") -> tuple[int, str]:
    """Return ``(resolution, note)``, snapping instead of raising.

    Used by the notebook so a single ``SUP_RES`` value can drive both ResNet and
    ViT arms: each arm silently gets a resolution it can actually consume, and
    the note is printed so the change is never invisible.
    """
    patch = patch_size_for(backbone)
    if is_compatible(requested, patch):
        return requested, "ok"
    snapped = snap_to_patch(requested, patch, mode)
    note = (
        f"{requested} -> {snapped} (snapped {mode}: '{backbone}' needs a multiple of {patch})"
    )
    return snapped, note


def num_patch_tokens(resolution: int, patch: int | None) -> int:
    """Token count fed to attention, excluding the CLS token.

    Handy for memory budgeting: attention cost grows with the square of this.
    """
    if patch is None:
        return 0
    assert_compatible(resolution, f"generic{patch}")
    side = resolution // patch
    return side * side


# ============================================================================
# data_resolver.py  --  inlined from src/dermssl/data_resolver.py (do not edit here)
# ============================================================================

"""Dataset resolution with three interchangeable backends.

Why this module exists
----------------------
The v4 notebook hardcoded ``/kaggle/input``. That path only exists inside a
Kaggle session, so the exact same notebook could not run on a university
cluster. This module removes that coupling: the notebook asks for datasets by
name and gets back local paths, no matter where it runs.

Backends
--------
``kaggle-input``
    Datasets already mounted by Kaggle under ``/kaggle/input``. Offline, instant.
``kagglehub``
    Datasets downloaded on demand with the ``kagglehub`` package. Works on any
    machine with credentials (``KAGGLE_USERNAME`` + ``KAGGLE_KEY``, or
    ``~/.kaggle/kaggle.json``). Inside Kaggle it transparently attaches the
    input instead of downloading.
``local``
    Copies already staged on disk (a cluster scratch directory, an NFS mount).

``auto`` resolves in that order of preference and reports which one it picked.

All discovery helpers take an explicit list of search roots so they can be unit
tested against temporary directories without any cloud access.
"""


import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, Sequence

KAGGLE_INPUT = Path("/kaggle/input")

# Kaggle dataset handles used by the project. Datasets themselves are NOT
# changing; only the way we reach them does.
DATASET_HANDLES = {
    "ham10000": "kmader/skin-cancer-mnist-ham10000",
    "isic2019": "nischaydnk/isic-2019-jpg-224x224-resized",
    "isic2020": "nischaydnk/isic-2020-jpg-256x256-resized",
}

BACKENDS = ("kaggle-input", "kagglehub", "local")


class DataResolutionError(RuntimeError):
    """Raised when a required dataset cannot be located in any backend."""


@dataclass
class ResolvedData:
    """Everything downstream cells need, with provenance attached."""

    backend: str
    roots: list[Path] = field(default_factory=list)
    ham_metadata_csv: Path | None = None
    ham_images: dict[str, Path] = field(default_factory=dict)
    pool_images: dict[str, Path] = field(default_factory=dict)
    pool_indexed: bool = True
    candidate_label_csvs: list[Path] = field(default_factory=list)

    def summary(self) -> dict:
        return {
            "backend": self.backend,
            "roots": [str(r) for r in self.roots],
            "ham_metadata_csv": str(self.ham_metadata_csv) if self.ham_metadata_csv else None,
            "ham_images": len(self.ham_images),
            # Distinguish "indexed and found none" from "deliberately not indexed".
            "pool_images": len(self.pool_images) if self.pool_indexed else "no indexado",
            "candidate_label_csvs": [str(p) for p in self.candidate_label_csvs],
        }


def detect_backend(
    kaggle_input: Path = KAGGLE_INPUT,
    local_root: Path | None = None,
) -> str:
    """Pick a backend without touching the network.

    Preference: mounted Kaggle inputs > staged local copies > kagglehub download.
    """
    if kaggle_input.is_dir() and any(kaggle_input.iterdir()):
        return "kaggle-input"
    if local_root is not None and Path(local_root).is_dir():
        if any(Path(local_root).rglob("HAM10000_metadata.csv")):
            return "local"
    return "kagglehub"


def index_images(roots: Iterable[Path], pattern: str = "**/*.jpg") -> dict[str, Path]:
    """Map image stem -> path for every match under ``roots``.

    Later roots win on collisions, which is why callers pass the authoritative
    root last. Stems are unique per dataset in ISIC/HAM, so collisions mean the
    same image is mounted twice.
    """
    found: dict[str, Path] = {}
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for path in root.glob(pattern):
            if path.is_file():
                found[path.stem] = path
    return found


def find_files(roots: Iterable[Path], filename: str) -> list[Path]:
    """Locate every file called ``filename`` under any of ``roots``."""
    hits: list[Path] = []
    for root in roots:
        root = Path(root)
        if root.is_dir():
            hits.extend(sorted(root.rglob(filename)))
    return hits


def find_label_csvs(roots: Iterable[Path], keyword: str = "groundtruth") -> list[Path]:
    """Find candidate ground-truth CSVs (external validation labels).

    Matching is case-insensitive on the file name so ``ISIC_2019_Training_
    GroundTruth.csv`` and ``groundtruth.csv`` both match.
    """
    hits: list[Path] = []
    key = keyword.lower()
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for path in sorted(root.rglob("*.csv")):
            if key in path.name.lower():
                hits.append(path)
    return hits


def _kagglehub_download(handle: str, output_dir: Path | None) -> Path:
    """Download one dataset through kagglehub. Imported lazily on purpose.

    Lazy import keeps this module usable (and testable) on machines that never
    installed kagglehub.
    """
    try:
        import kagglehub  # noqa: PLC0415 - lazy on purpose
    except ImportError as exc:  # pragma: no cover - environment dependent
        raise DataResolutionError(
            "backend 'kagglehub' requires the kagglehub package: pip install kagglehub"
        ) from exc

    kwargs = {}
    if output_dir is not None:
        kwargs["output_dir"] = str(output_dir)
    return Path(kagglehub.dataset_download(handle, **kwargs))


def credentials_present() -> bool:
    """True when kagglehub has some way to authenticate.

    Checked before downloading so the failure message is about credentials
    instead of an opaque HTTP 401 several minutes in.
    """
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    if os.environ.get("KAGGLE_API_TOKEN"):
        return True
    return (Path.home() / ".kaggle" / "kaggle.json").is_file()


def dataset_roots(
    backend: str,
    datasets: Sequence[str],
    kaggle_input: Path = KAGGLE_INPUT,
    local_root: Path | None = None,
    cache_dir: Path | None = None,
) -> list[Path]:
    """Return the filesystem roots to search, per backend."""
    if backend not in BACKENDS:
        raise ValueError(f"unknown backend {backend!r}; expected one of {BACKENDS}")

    if backend == "kaggle-input":
        if not Path(kaggle_input).is_dir():
            raise DataResolutionError(f"{kaggle_input} does not exist; are you inside Kaggle?")
        return [Path(kaggle_input)]

    if backend == "local":
        if local_root is None:
            raise DataResolutionError("backend 'local' needs local_root=<path>")
        root = Path(local_root)
        if not root.is_dir():
            raise DataResolutionError(f"local_root {root} does not exist")
        return [root]

    if not credentials_present():
        raise DataResolutionError(
            "backend 'kagglehub' found no credentials. Set KAGGLE_USERNAME + KAGGLE_KEY "
            "or place ~/.kaggle/kaggle.json (chmod 600). Never commit that file."
        )
    roots = []
    for name in datasets:
        handle = DATASET_HANDLES.get(name, name)
        target = None if cache_dir is None else Path(cache_dir) / name
        roots.append(_kagglehub_download(handle, target))
    return roots


def resolve_datasets(
    backend: str = "auto",
    datasets: Sequence[str] = ("ham10000", "isic2019", "isic2020"),
    kaggle_input: Path = KAGGLE_INPUT,
    local_root: Path | None = None,
    cache_dir: Path | None = None,
    index_pool: bool = True,
) -> ResolvedData:
    """Locate HAM10000 plus the unlabeled pool, whatever the environment.

    ``index_pool=False`` skips building the path index for the unlabelled pool.
    That index costs real time: on Kaggle, walking the 48 442 ISIC images took
    157 of the 192 seconds of a smoke run - and the frozen-backbone evaluation
    never touches the pool. Only the continual pre-training stage needs it, so it
    is opt-in per stage rather than always paid.

    Returns a :class:`ResolvedData` describing what was found. Raises
    :class:`DataResolutionError` with an actionable message when HAM10000 (the
    only mandatory dataset) is missing.
    """
    if backend == "auto":
        backend = detect_backend(kaggle_input=kaggle_input, local_root=local_root)

    roots = dataset_roots(
        backend,
        datasets,
        kaggle_input=kaggle_input,
        local_root=local_root,
        cache_dir=cache_dir,
    )

    metadata_hits = find_files(roots, "HAM10000_metadata.csv")
    if not metadata_hits:
        raise DataResolutionError(
            "HAM10000_metadata.csv not found. On Kaggle: Add Input -> "
            "'Skin Cancer MNIST: HAM10000' (kmader). Elsewhere: use "
            "backend='kagglehub' or stage the dataset and pass local_root."
        )
    ham_metadata_csv = metadata_hits[0]
    ham_dir = ham_metadata_csv.parent

    ham_images = index_images([ham_dir])
    pool_images: dict[str, Path] = {}
    if index_pool:
        # Pool = every other jpg reachable from the roots, minus the HAM images.
        all_images = index_images(roots)
        pool_images = {k: v for k, v in all_images.items() if k not in ham_images}

    return ResolvedData(
        backend=backend,
        roots=[Path(r) for r in roots],
        ham_metadata_csv=ham_metadata_csv,
        ham_images=ham_images,
        pool_images=pool_images,
        pool_indexed=index_pool,
        candidate_label_csvs=find_label_csvs(roots),
    )


# ============================================================================
# label_regimes.py  --  inlined from src/dermssl/label_regimes.py (do not edit here)
# ============================================================================

"""Few-label regime sampling for frozen-backbone probes.

Why this module exists
----------------------
The whole point of self-supervised learning is that it should need *fewer*
labels. To measure that you evaluate the same frozen backbone while giving the
classifier 1 %, 5 %, 10 %, ... of the training labels. v4 started at 10 %, which
skips the regimes where SSL usually wins by the widest margin.

Two properties matter and neither is free:

**Every class must survive.** ``df`` (dermatofibroma) has roughly 80 training
images. One percent of the whole training set naively sampled can leave ``df``
with zero examples, and a classifier that never saw a class cannot predict it -
macro-F1 collapses for a reason that has nothing to do with the backbone.
``min_per_class`` prevents that.

**Regimes must nest.** The 1 % subset should be contained in the 5 % subset,
which is contained in 10 %, and so on. Nesting means the only difference between
two regimes is *added* labels, so the resulting curve measures label efficiency
instead of sampling noise. Achieved by shuffling each class once per seed and
taking prefixes.
"""


import numpy as np

DEFAULT_FRACTIONS = (0.01, 0.05, 0.10, 0.20, 0.50, 1.00)


class RegimeError(ValueError):
    """Raised when a requested regime cannot be satisfied."""


def _class_order(y: np.ndarray, seed: int) -> dict[int, np.ndarray]:
    """Per-class index list, shuffled deterministically from ``seed``.

    The same seed always yields the same order, which is what makes prefixes
    nest across fractions.
    """
    rng = np.random.RandomState(seed)
    order: dict[int, np.ndarray] = {}
    for cls in np.unique(y):
        idx = np.flatnonzero(y == cls)
        order[int(cls)] = idx[rng.permutation(len(idx))]
    return order


def regime_size(n_class: int, fraction: float, min_per_class: int) -> int:
    """How many samples one class contributes to a regime.

    Never more than the class actually has, never fewer than
    ``min_per_class`` (unless the class is smaller than that).
    """
    if not 0.0 < fraction <= 1.0:
        raise RegimeError(f"fraction must be in (0, 1], got {fraction}")
    wanted = int(round(n_class * fraction))
    wanted = max(wanted, min(min_per_class, n_class))
    return min(wanted, n_class)


def stratified_regime_indices(
    y: np.ndarray,
    fraction: float,
    seed: int = 42,
    min_per_class: int = 2,
) -> np.ndarray:
    """Indices of a class-balanced-by-proportion subset of ``y``.

    Returns sorted, duplicate-free indices into ``y``. Guarantees that every
    class present in ``y`` is also present in the result.
    """
    y = np.asarray(y)
    if y.ndim != 1 or y.size == 0:
        raise RegimeError("y must be a non-empty 1-D array of class indices")

    order = _class_order(y, seed)
    picked: list[np.ndarray] = []
    for cls, idx in order.items():
        take = regime_size(len(idx), fraction, min_per_class)
        picked.append(idx[:take])
    return np.sort(np.concatenate(picked))


def build_regimes(
    y: np.ndarray,
    fractions: tuple[float, ...] = DEFAULT_FRACTIONS,
    seed: int = 42,
    min_per_class: int = 2,
) -> dict[float, np.ndarray]:
    """All regimes for one seed, nested by construction.

    ``build_regimes(y)[0.01]`` is a subset of ``build_regimes(y)[0.05]`` because
    both are prefixes of the same per-class shuffle.
    """
    return {
        f: stratified_regime_indices(y, f, seed=seed, min_per_class=min_per_class)
        for f in sorted(fractions)
    }


def describe_regimes(y: np.ndarray, regimes: dict[float, np.ndarray]) -> list[dict]:
    """Human-readable audit of what each regime actually contains.

    Printed in the notebook so a reader can confirm no class was starved.
    """
    y = np.asarray(y)
    classes = np.unique(y)
    rows = []
    for frac in sorted(regimes):
        idx = regimes[frac]
        counts = {int(c): int((y[idx] == c).sum()) for c in classes}
        rows.append(
            {
                "fraction": frac,
                "n_labels": int(len(idx)),
                "pct_of_train": round(100.0 * len(idx) / len(y), 2),
                "per_class": counts,
                "min_class_count": min(counts.values()),
            }
        )
    return rows


# ============================================================================
# metrics.py  --  inlined from src/dermssl/metrics.py (do not edit here)
# ============================================================================

"""Evaluation metrics implemented with numpy only.

Why not scikit-learn
--------------------
scikit-learn is available on Kaggle but not on every machine, and these five
metrics are short enough to own. Owning them buys two things: the notebook can
be unit tested locally without extra dependencies, and the definition of every
reported number is visible in the repository instead of hidden behind a version
of a library.

Definitions used
----------------
``macro_f1``
    F1 computed per class, then averaged unweighted. Chosen over accuracy
    because HAM10000 is imbalanced ~58:1 - accuracy is dominated by ``nv``.
``auc_ovr_macro``
    One-vs-rest ROC AUC per class, averaged. Rank based (Mann-Whitney U) with
    proper handling of tied scores.
``ece``
    Expected Calibration Error: |confidence - accuracy| averaged over
    equal-width confidence bins, weighted by bin population. Answers "when the
    model says 90 %, is it right 90 % of the time?".
"""


import numpy as np


def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if y_true.size == 0:
        return float("nan")
    return float((y_true == y_pred).mean())


def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> np.ndarray:
    """Rows are the true class, columns the predicted class."""
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(np.asarray(y_true), np.asarray(y_pred)):
        cm[int(t), int(p)] += 1
    return cm


def per_class_f1(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> np.ndarray:
    """F1 per class; classes with no true and no predicted samples score 0."""
    cm = confusion_matrix(y_true, y_pred, n_classes)
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    denom = 2 * tp + fp + fn
    with np.errstate(divide="ignore", invalid="ignore"):
        f1 = np.where(denom > 0, 2 * tp / denom, 0.0)
    return f1


def macro_f1(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> float:
    return float(per_class_f1(y_true, y_pred, n_classes).mean())


def recall_per_class(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> np.ndarray:
    cm = confusion_matrix(y_true, y_pred, n_classes)
    support = cm.sum(axis=1).astype(float)
    tp = np.diag(cm).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(support > 0, tp / support, 0.0)


def precision_per_class(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> np.ndarray:
    """Precision per class; classes never predicted score 0, not NaN."""
    cm = confusion_matrix(y_true, y_pred, n_classes)
    predicted = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(predicted > 0, tp / predicted, 0.0)


def macro_precision(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> float:
    return float(precision_per_class(y_true, y_pred, n_classes).mean())


def macro_recall(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> float:
    return float(recall_per_class(y_true, y_pred, n_classes).mean())


def balanced_accuracy(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> float:
    """Mean per-class recall: immune to class imbalance, unlike plain accuracy.

    Reference anchor: the official ISIC 2019 winner scored 63.6% on this
    metric; plain accuracy on HAM10000 can hit 67% by always predicting nv.
    """
    return macro_recall(y_true, y_pred, n_classes)


def _average_ranks(x: np.ndarray) -> np.ndarray:
    """Ranks starting at 1, with tied values sharing their average rank.

    Tie handling is not cosmetic: a model that outputs the same probability for
    many samples would get a wrongly optimistic AUC with naive ranking.
    """
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    sorted_x = x[order]
    ranks = np.empty(len(x), dtype=float)
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and sorted_x[j + 1] == sorted_x[i]:
            j += 1
        ranks[order[i : j + 1]] = 0.5 * (i + j) + 1.0
        i = j + 1
    return ranks


def binary_auc(y_binary: np.ndarray, scores: np.ndarray) -> float:
    """ROC AUC for one class via the Mann-Whitney U statistic.

    Returns NaN when the class has only positives or only negatives, because
    AUC is undefined there (and silently returning 0.5 would hide the problem).
    """
    y_binary = np.asarray(y_binary).astype(int)
    n_pos = int(y_binary.sum())
    n_neg = int(len(y_binary) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = _average_ranks(np.asarray(scores, dtype=float))
    rank_sum = ranks[y_binary == 1].sum()
    return float((rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def auc_ovr_macro(y_true: np.ndarray, proba: np.ndarray) -> float:
    """Macro-averaged one-vs-rest AUC. Undefined classes are skipped."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba, dtype=float)
    aucs = [binary_auc((y_true == c).astype(int), proba[:, c]) for c in range(proba.shape[1])]
    valid = [a for a in aucs if a == a]  # drop NaN
    return float(np.mean(valid)) if valid else float("nan")


def ece(y_true: np.ndarray, proba: np.ndarray, n_bins: int = 15) -> float:
    """Expected Calibration Error over equal-width confidence bins."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba, dtype=float)
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        total += mask.mean() * abs(conf[mask].mean() - correct[mask].mean())
    return float(total)


def summarize(y_true: np.ndarray, proba: np.ndarray, n_classes: int) -> dict:
    """One call, every headline metric. Used by every probe."""
    pred = np.asarray(proba).argmax(axis=1)
    return {
        "macro_f1": macro_f1(y_true, pred, n_classes),
        "accuracy": accuracy(y_true, pred),
        "balanced_accuracy": balanced_accuracy(y_true, pred, n_classes),
        "macro_precision": macro_precision(y_true, pred, n_classes),
        "macro_recall": macro_recall(y_true, pred, n_classes),
        "auc": auc_ovr_macro(y_true, proba),
        "ece": ece(y_true, proba),
    }


def bootstrap_ci(
    y_true: np.ndarray,
    proba: np.ndarray,
    metric_fn,
    n_resamples: int = 1000,
    seed: int = 42,
    alpha: float = 0.05,
) -> tuple[float, float, float]:
    """Percentile bootstrap CI: ``(point_estimate, low, high)``.

    Resamples the validation set with replacement. Reporting an interval instead
    of a bare number is the difference between "0.69" and "0.69, and with this
    sample size it could plausibly be 0.66 to 0.73".
    """
    y_true = np.asarray(y_true)
    proba = np.asarray(proba, dtype=float)
    point = metric_fn(y_true, proba)
    rng = np.random.RandomState(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_resamples):
        idx = rng.randint(0, n, n)
        v = metric_fn(y_true[idx], proba[idx])
        if v == v:
            vals.append(v)
    if not vals:
        return float(point), float("nan"), float("nan")
    lo = float(np.percentile(vals, 100 * alpha / 2))
    hi = float(np.percentile(vals, 100 * (1 - alpha / 2)))
    return float(point), lo, hi


FST_BUCKETS = (1, 2, 3, 4, 5, 6)
UNAVAILABLE_BUCKET = "unavailable"


def stratified_summary(
    y_true: np.ndarray,
    proba: np.ndarray,
    buckets: np.ndarray,
    n_classes: int,
    min_n: int = 30,
) -> dict:
    """``summarize()``, split per Fitzpatrick skin-type bucket (1-6, or
    ``"unavailable"`` for ``FSTEstimate.bucket is None`` rows - see
    ``dermssl.fitzpatrick``).

    Never fails on thin data (spec.md, fairness-evaluation req 2): every
    bucket in ``FST_BUCKETS + (UNAVAILABLE_BUCKET,)`` is always present in the
    output, even with zero matching rows - a bucket below ``min_n`` (0
    included) is marked ``"SKIPPED"`` with the observed count and a reason,
    never silently omitted from the structure.

    With no rows at all (``y_true`` empty - e.g. no ``fst_manifest.csv`` has
    been produced yet), returns ``{}``: nothing to stratify, matching
    design.md's error table ("Manifest missing -> stratified_summary returns
    {}; gate = SKIP").
    """
    y_true = np.asarray(y_true)
    if y_true.size == 0:
        return {}

    proba = np.asarray(proba, dtype=float)
    buckets = np.asarray(buckets, dtype=object)

    result: dict = {}
    for key in (*FST_BUCKETS, UNAVAILABLE_BUCKET):
        if key == UNAVAILABLE_BUCKET:
            mask = np.array([b is None for b in buckets])
        else:
            mask = np.array([b == key for b in buckets])
        count = int(mask.sum())
        if count < min_n:
            result[key] = {
                "status": "SKIPPED",
                "count": count,
                "reason": f"n={count} < min_n={min_n}",
            }
        else:
            result[key] = {
                "status": "OK",
                "count": count,
                **summarize(y_true[mask], proba[mask], n_classes),
            }
    return result


DEFAULT_COMPARISON_METRIC = "macro_f1"
DEFAULT_COMPARISON_MIN_BUCKET_N = 30


def _variant_gap(summary: dict, *, metric: str, min_bucket_n: int) -> float | None:
    """Max-minus-min ``metric`` among buckets with ``status == "OK"`` and at
    least ``min_bucket_n`` samples - the same qualifying rule
    ``gates.fairness_gate`` applies, reimplemented here (not imported) so
    ``metrics.py`` stays independent of ``gates.py`` (mirrors D1: neither
    module depends on the other; the caller composes them).

    Returns ``None`` when fewer than two buckets qualify - "insufficient
    data", not zero: a report of ``0.0`` would misleadingly claim a perfect
    (measured) fairness gap where there is, in fact, nothing to measure.
    """
    qualifying = {
        bucket: entry[metric]
        for bucket, entry in summary.items()
        if entry.get("status") == "OK" and entry.get("count", 0) >= min_bucket_n
    }
    if len(qualifying) < 2:
        return None
    return max(qualifying.values()) - min(qualifying.values())


def comparison_report(
    baseline: dict,
    regularization: dict,
    synthetic_blend: dict,
    *,
    metric: str = DEFAULT_COMPARISON_METRIC,
    min_bucket_n: int = DEFAULT_COMPARISON_MIN_BUCKET_N,
) -> dict:
    """Stage 2 baseline-vs-mitigation comparison (spec.md, circle-adaptation
    req 5): consumes three ``stratified_summary()`` outputs (baseline,
    regularization-only, synthetic-blend) and reports the per-FST-bucket gap
    for each, plus each mitigation variant's relative improvement over the
    baseline gap - the thesis's primary acceptance criterion
    (proposal.md, "Success Criteria": "measurable relative fairness
    improvement ... of Stage 2 over the Stage 1 baseline").

    This function computes arithmetic over whatever three summaries it is
    given - it does not run any training or produce any data itself. As of
    this writing no real Stage 1 or Stage 2 run has happened (see
    docs/21-checkpoint-manual-circle-stage2.md), so calling this with real
    data is future work; it is tested here against synthetic,
    hand-built ``stratified_summary()``-shaped fixtures only.

    ``relative_improvement`` is ``(baseline_gap - variant_gap) / baseline_gap``:
    positive means the variant's gap is smaller (fairer) than the baseline's.
    It is ``None`` - never a number - whenever it cannot be computed without
    guessing: the baseline gap is undefined (fewer than two qualifying
    buckets), the baseline gap is exactly zero (division by zero), or the
    variant's own gap is undefined.
    """
    baseline_gap = _variant_gap(baseline, metric=metric, min_bucket_n=min_bucket_n)
    report = {
        "baseline": {
            "gap": baseline_gap,
            "status": "ok" if baseline_gap is not None else "insufficient_data",
        }
    }
    for name, summary in (("regularization", regularization), ("synthetic_blend", synthetic_blend)):
        gap = _variant_gap(summary, metric=metric, min_bucket_n=min_bucket_n)
        relative_improvement = None
        if gap is not None and baseline_gap is not None and baseline_gap != 0:
            relative_improvement = (baseline_gap - gap) / baseline_gap
        report[name] = {
            "gap": gap,
            "status": "ok" if gap is not None else "insufficient_data",
            "relative_improvement": relative_improvement,
        }
    return report


def mean_std(values) -> dict:
    """Mean and population std over seeds, NaN-safe."""
    vals = [float(v) for v in values if v == v]
    if not vals:
        return {"mean": float("nan"), "std": float("nan"), "n": 0}
    return {
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "n": len(vals),
    }


# ============================================================================
# probes.py  --  inlined from src/dermssl/probes.py (do not edit here)
# ============================================================================

"""Frozen-backbone probes: how we evaluate without ever training the backbone.

Why this module exists
----------------------
The claim of the thesis is "no label ever touched the feature extractor". To
keep that claim true, evaluation must read the frozen features and fit only a
tiny classifier on top. Three probes, in increasing number of trained
parameters:

``knn_predict_proba``
    **Zero trained parameters.** Classify a validation image by the labels of
    its nearest training neighbours in feature space (cosine distance). This is
    the cleanest possible measure of representation quality: nothing is learned,
    so the number reflects the geometry of the features and nothing else.

``prototype_predict_proba``
    **Zero trained parameters.** One average feature vector per class; classify
    by closest average. Robust with very few labels, which is exactly the 1 %
    regime.

``logreg_predict_proba``
    **One linear layer.** Multinomial logistic regression, the standard "linear
    probe" of the SSL literature. Needs scikit-learn, imported lazily.

All three take features as plain numpy arrays, which is why the expensive part
(running images through the backbone) happens once and is cached.
"""


import numpy as np

CHUNK = 512  # rows of the validation set scored per matmul; bounds peak memory


def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Scale every row to unit length so a dot product *is* cosine similarity."""
    x = np.asarray(x, dtype=np.float32)
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norm, eps)


def _counts_to_proba(counts: np.ndarray) -> np.ndarray:
    """Normalise per row; rows that sum to zero become a uniform distribution."""
    total = counts.sum(axis=1, keepdims=True)
    uniform = np.full_like(counts, 1.0 / counts.shape[1])
    return np.where(total > 0, counts / np.maximum(total, 1e-12), uniform)


def effective_k(requested_k: int, n_train: int, n_classes: int = 0) -> int:
    """Clip ``k`` to the labelled set size. A crash guard, nothing more.

    What we measured, and why ``k`` is deliberately *not* shrunk further
    ----------------------------------------------------------------------
    The Kaggle run of 2026-07-29 reported accuracy 0.663 next to macro-F1 0.181
    for the k-NN probe. Reproducing it with the real HAM10000 class proportions
    (648 labels, 7 classes, so ``df`` gets ~10 samples against ``nv``'s ~440)
    confirmed the mechanism: with ``k=20`` the probe only ever predicts **2 of 7
    classes**, because a class with 10 labelled samples cannot out-vote a class
    with 440.

    The tempting fix - shrink ``k`` - was measured and rejected:

    ==== ======== ========= ====== =================
    k    accuracy macro-F1  AUC    classes predicted
    ==== ======== ========= ====== =================
    20   0.639    0.124     0.738  2
    10   0.651    0.170     0.698  4
    7    0.639    0.183     0.674  5
    5    0.627    0.209     0.575  5
    3    0.615    0.202     0.563  7
    ==== ======== ========= ====== =================

    Lowering ``k`` buys a little macro-F1 and destroys AUC. It is a worse trade,
    not a fix.

    The real conclusion is about *reporting*, not about ``k``: under 58:1
    imbalance the argmax of an unweighted neighbour vote is dominated by the
    class prior, so **k-NN macro-F1 is not a meaningful headline number**. Read
    AUC for k-NN (rank based, prior independent), read macro-F1 from a probe that
    can weight classes - see :func:`logreg_predict_proba` with
    ``class_weight='balanced'`` and :func:`knn_balanced_predict_proba`.
    """
    if n_train <= 0:
        raise ValueError("k-NN probe needs at least one labelled training sample")
    return int(max(1, min(requested_k, n_train)))


def knn_predict_proba(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    n_classes: int,
    k: int = 20,
    temperature: float = 0.07,
    balanced: bool = False,
) -> np.ndarray:
    """Temperature-weighted cosine k-NN, as used to evaluate DINO/DINOv2.

    Each of the ``k`` nearest training neighbours votes for its class with
    weight ``exp(similarity / temperature)``. A low temperature makes the closest
    neighbour dominate; 0.07 is the value from the DINO paper.

    ``k`` is reduced by :func:`effective_k` for small or imbalanced label
    budgets. With ``balanced=True`` each vote is divided by how often its class
    appears in the labelled subset, which removes the majority-class prior; that
    is reported as a **separate** probe rather than applied silently, because it
    is a different estimator and not a bug fix.
    """
    x_train = l2_normalize(x_train)
    x_val = l2_normalize(x_val)
    y_train = np.asarray(y_train)
    k = effective_k(k, len(x_train), n_classes)

    class_weight = np.ones(n_classes, dtype=np.float64)
    if balanced:
        counts = np.bincount(y_train, minlength=n_classes).astype(np.float64)
        class_weight = 1.0 / np.maximum(counts, 1.0)  # unseen classes keep weight 1

    proba = np.zeros((len(x_val), n_classes), dtype=np.float64)
    for start in range(0, len(x_val), CHUNK):
        block = x_val[start : start + CHUNK]
        sims = block @ x_train.T  # cosine similarity, both sides normalised
        top = np.argpartition(-sims, k - 1, axis=1)[:, :k]
        top_sims = np.take_along_axis(sims, top, axis=1)
        weights = np.exp(top_sims / temperature)
        for row in range(len(block)):
            np.add.at(proba[start + row], y_train[top[row]], weights[row])
    return _counts_to_proba(proba * class_weight)


def knn_balanced_predict_proba(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    n_classes: int,
    k: int = 20,
    temperature: float = 0.07,
) -> np.ndarray:
    """Class-balanced k-NN: same neighbours, votes divided by class frequency.

    Reported alongside plain k-NN so the effect of removing the majority-class
    prior is visible instead of assumed.
    """
    return knn_predict_proba(
        x_train, y_train, x_val, n_classes, k=k, temperature=temperature, balanced=True
    )


def prototype_predict_proba(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    n_classes: int,
    temperature: float = 0.1,
) -> np.ndarray:
    """Nearest-class-mean classifier in cosine space.

    Classes absent from ``y_train`` get a zero prototype, which yields a
    similarity of 0 - never selected over a real class, never crashing.
    """
    x_train = l2_normalize(x_train)
    x_val = l2_normalize(x_val)
    y_train = np.asarray(y_train)

    protos = np.zeros((n_classes, x_train.shape[1]), dtype=np.float32)
    for cls in range(n_classes):
        mask = y_train == cls
        if mask.any():
            protos[cls] = x_train[mask].mean(axis=0)
    protos = l2_normalize(protos)

    logits = (x_val @ protos.T) / temperature
    logits -= logits.max(axis=1, keepdims=True)  # stabilise before exp
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def logreg_predict_proba(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    n_classes: int,
    max_iter: int = 2000,
    c: float = 1.0,
    seed: int = 42,
    class_weight: str | None = None,
) -> np.ndarray:
    """Standard linear probe. Requires scikit-learn (present on Kaggle).

    Features are standardised first: logistic regression converges far better on
    zero-mean unit-variance inputs, and ViT feature scales vary a lot by layer.
    Columns absent from ``y_train`` are filled with zero probability so the
    output always has ``n_classes`` columns.

    ``class_weight='balanced'`` reweights the loss by inverse class frequency.
    This is the probe where macro-F1 *is* meaningful under heavy imbalance, since
    the estimator itself can be told not to follow the class prior - unlike an
    unweighted neighbour vote (see :func:`effective_k`).
    """
    try:
        from sklearn.linear_model import LogisticRegression  # noqa: PLC0415
        from sklearn.preprocessing import StandardScaler  # noqa: PLC0415
    except ImportError as exc:  # pragma: no cover - environment dependent
        raise RuntimeError("logreg probe needs scikit-learn: pip install scikit-learn") from exc

    scaler = StandardScaler().fit(x_train)
    model = LogisticRegression(
        max_iter=max_iter, C=c, random_state=seed, class_weight=class_weight
    )
    model.fit(scaler.transform(x_train), y_train)
    partial = model.predict_proba(scaler.transform(x_val))

    full = np.zeros((len(x_val), n_classes), dtype=np.float64)
    for col, cls in enumerate(model.classes_):
        full[:, int(cls)] = partial[:, col]
    return full


def logreg_balanced_predict_proba(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    n_classes: int,
    max_iter: int = 2000,
    c: float = 1.0,
    seed: int = 42,
) -> np.ndarray:
    """Linear probe with inverse-frequency class weights.

    Reported next to the unweighted version rather than replacing it: whether the
    reweighting helps macro-F1 on these features is an empirical question, and
    the run answers it. We do not presume the answer.
    """
    return logreg_predict_proba(
        x_train, y_train, x_val, n_classes,
        max_iter=max_iter, c=c, seed=seed, class_weight="balanced",
    )


PROBES = {
    "knn": knn_predict_proba,
    "knn_balanced": knn_balanced_predict_proba,
    "prototype": prototype_predict_proba,
    "logreg": logreg_predict_proba,
    "logreg_balanced": logreg_balanced_predict_proba,
}

# Probes whose scores are similarity-derived rather than fitted probabilities.
# Their ECE is not meaningful: the softmax temperature is an arbitrary choice, not
# a calibrated one. Reported for completeness, never used as a calibration claim.
UNCALIBRATED_PROBES = ("knn", "knn_balanced", "prototype")

# Probes whose argmax follows the class prior under heavy imbalance, so their
# macro-F1 must not be quoted as a headline result. See effective_k() for the
# measurements behind this. AUC stays valid for all probes: it is rank based.
PRIOR_DOMINATED_PROBES = ("knn", "prototype")


def headline_metric(probe: str) -> str:
    """Which metric may be quoted as this probe's result."""
    return "auc" if probe in PRIOR_DOMINATED_PROBES else "macro_f1"


def run_probe(
    name: str,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    n_classes: int,
    **kwargs,
) -> np.ndarray:
    """Dispatch by probe name so the notebook can loop over probes by string."""
    if name not in PROBES:
        raise ValueError(f"unknown probe {name!r}; expected one of {sorted(PROBES)}")
    return PROBES[name](x_train, y_train, x_val, n_classes, **kwargs)


# ============================================================================
# feature_cache.py  --  inlined from src/dermssl/feature_cache.py (do not edit here)
# ============================================================================

"""Persistent cache for extracted features.

Why this module exists
----------------------
Running 10 000 images through a ViT-B is the expensive part of the evaluation -
minutes of GPU time. Fitting a probe on the resulting vectors is milliseconds.
So we extract features **once**, write them to disk, and then run as many
probes, label regimes and seeds as we like for free. That is what turns "5 seeds
x 6 regimes x 3 probes" from an impossible bill into a rounding error, and it is
what lets a Kaggle session survive a restart without redoing work.

The safety property that matters: a cache must never return features that belong
to a different configuration. The key therefore includes a fingerprint of the
*ordered* list of image ids, because features are stored aligned to that order.
Change the split, and the key changes, and the stale cache is ignored instead of
silently corrupting every downstream number.
"""


import hashlib
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import numpy as np

SCHEMA_VERSION = 1


def fingerprint_ids(ids: Sequence[str]) -> str:
    """Short, order-sensitive digest of an id list."""
    digest = hashlib.sha256()
    for i in ids:
        digest.update(str(i).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()[:16]


def cache_key(
    backbone: str,
    split: str,
    resolution: int,
    ids: Sequence[str],
    transform_tag: str = "eval",
) -> str:
    """Filesystem-safe key identifying one exact extraction configuration."""
    safe_backbone = backbone.replace("/", "-")
    return (
        f"v{SCHEMA_VERSION}__{safe_backbone}__{split}__r{resolution}"
        f"__{transform_tag}__{fingerprint_ids(ids)}"
    )


@dataclass
class CachedFeatures:
    key: str
    features: np.ndarray
    labels: np.ndarray
    meta: dict


class FeatureCache:
    """Directory-backed store of ``(features, labels, meta)`` triples."""

    def __init__(self, root: Path):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    def paths_for(self, key: str) -> tuple[Path, Path, Path]:
        base = self.root / key
        return (
            base.with_suffix(".features.npy"),
            base.with_suffix(".labels.npy"),
            base.with_suffix(".meta.json"),
        )

    def has(self, key: str) -> bool:
        return all(p.is_file() for p in self.paths_for(key))

    def save(
        self,
        key: str,
        features: np.ndarray,
        labels: np.ndarray,
        meta: dict | None = None,
    ) -> CachedFeatures:
        """Write one entry. Rejects mismatched lengths instead of storing garbage."""
        features = np.asarray(features, dtype=np.float32)
        labels = np.asarray(labels)
        if len(features) != len(labels):
            raise ValueError(
                f"features/labels length mismatch: {len(features)} vs {len(labels)}"
            )
        f_path, l_path, m_path = self.paths_for(key)
        full_meta = {
            "key": key,
            "schema": SCHEMA_VERSION,
            "n_samples": int(len(features)),
            "dim": int(features.shape[1]) if features.ndim == 2 else None,
            **(meta or {}),
        }
        np.save(f_path, features)
        np.save(l_path, labels)
        m_path.write_text(json.dumps(full_meta, indent=2), encoding="utf-8")
        return CachedFeatures(key, features, labels, full_meta)

    def load(self, key: str) -> CachedFeatures | None:
        """Return the entry, or None when absent or inconsistent.

        Returning None (rather than raising) is deliberate: the caller's natural
        reaction is "then extract it", which is always correct.
        """
        f_path, l_path, m_path = self.paths_for(key)
        if not all(p.is_file() for p in (f_path, l_path, m_path)):
            return None
        try:
            features = np.load(f_path)
            labels = np.load(l_path)
            meta = json.loads(m_path.read_text(encoding="utf-8"))
        except (OSError, ValueError, json.JSONDecodeError):
            return None
        if len(features) != len(labels):
            return None
        if meta.get("schema") != SCHEMA_VERSION:
            return None
        if meta.get("n_samples") not in (None, len(features)):
            return None
        return CachedFeatures(key, features, labels, meta)

    def get_or_extract(
        self,
        key: str,
        extract_fn,
        meta: dict | None = None,
    ) -> CachedFeatures:
        """Load ``key`` if cached, otherwise call ``extract_fn() -> (X, y)``.

        This single call is what makes the notebook resume-safe: a session that
        dies after extracting the training features re-uses them on restart.
        """
        cached = self.load(key)
        if cached is not None:
            return cached
        features, labels = extract_fn()
        return self.save(key, features, labels, meta=meta)

    def manifest(self) -> list[dict]:
        """Every entry currently on disk. Printed as an audit trail."""
        entries = []
        for m_path in sorted(self.root.glob("*.meta.json")):
            try:
                entries.append(json.loads(m_path.read_text(encoding="utf-8")))
            except json.JSONDecodeError:
                continue
        return entries


# ============================================================================
# fitzpatrick.py  --  inlined from src/dermssl/fitzpatrick.py (do not edit here)
# ============================================================================

"""Mask-guided ITA-based Fitzpatrick Skin Type (FST) estimation.

Why this module exists
-----------------------
The repo documents skin-tone bias as a known limit (HAM10000 skews FST
I-III) but never measures it. This module turns that limit into a number:
every image gets an FST bucket (1-6) or an explicit "unavailable" marker,
computed from skin pixels only (the lesion is excluded via a mask).

Pure numpy, no scikit-image at runtime (D2 in design.md). ``scikit-image``
is a *test oracle* only, used in ``tests/test_fitzpatrick.py`` to check the
LAB conversion against ``skimage.color.rgb2lab`` within ``atol=1e-4`` -
never imported here. ``src/dermssl/__init__.py::MODULE_ORDER`` modules are
inlined and executed by the notebook-build test, so a module-level
``import skimage`` here would break both that test and the Kaggle cell.
"""


from dataclasses import dataclass

import numpy as np

from dermssl.pretrain import lesion_guided

# Groh et al. empirical ITA thresholds (degrees), descending. Bucket 1 is the
# lightest FST, bucket 6 the darkest.
FST_THRESHOLDS = (55.0, 41.0, 28.0, 19.0, 10.0)

# Below this fraction of skin pixels, the estimate is not trustworthy.
MIN_SKIN_FRACTION = 0.05

# Mirrors pretrain.lesion_guided's degenerate-mask bounds: outside this
# range a mask is not a plausible single-lesion segmentation.
MIN_MASK_AREA_FRACTION = 0.01
MAX_MASK_AREA_FRACTION = 0.90

# Near-black pixels are treated as image border/vignette artifacts (common
# in dermoscopy captures), not skin - excluded regardless of the lesion mask
# (spec.md: "Skin Pixel Selection" MUST exclude border/vignette too).
BORDER_LUMINANCE_THRESHOLD = 0.02

# D65 reference white, 2-degree observer. Matches skimage's
# ``xyz_tristimulus_values(illuminant="D65", observer="2")`` exactly so the
# LAB oracle test can hold at atol=1e-4.
_D65_XN = 0.95047
_D65_YN = 1.0
_D65_ZN = 1.08883

# sRGB -> XYZ (D65). Matches skimage.color.colorconv.xyz_from_rgb exactly.
_SRGB_TO_XYZ = np.array(
    [
        [0.412453, 0.357580, 0.180423],
        [0.212671, 0.715160, 0.072169],
        [0.019334, 0.119193, 0.950227],
    ]
)


def _inverse_gamma(channel: np.ndarray) -> np.ndarray:
    """sRGB companding removal: gamma-encoded [0, 1] -> linear-light [0, 1]."""
    linear_low = channel / 12.92
    linear_high = ((channel + 0.055) / 1.055) ** 2.4
    return np.where(channel <= 0.04045, linear_low, linear_high)


def _f(t: np.ndarray) -> np.ndarray:
    """CIE Lab nonlinearity, piecewise cube-root with a linear floor near 0.

    Uses the same rounded constants as ``skimage.color.xyz2lab`` (0.008856,
    7.787, 16/116) rather than the more "exact" (6/29)**3 form, so the two
    implementations agree within the LAB oracle test's ``atol=1e-4``.
    """
    cube_root = np.cbrt(t)
    linear = 7.787 * t + 16.0 / 116.0
    return np.where(t > 0.008856, cube_root, linear)


def srgb_to_lab(img: np.ndarray) -> np.ndarray:
    """Convert an sRGB image, float in [0, 1], shape (H, W, 3), to CIE Lab.

    D65 illuminant, 2-degree observer - matches ``skimage.color.rgb2lab``
    within ``atol=1e-4`` (see ``tests/test_fitzpatrick.py``).
    """
    linear = _inverse_gamma(img)
    xyz = linear @ _SRGB_TO_XYZ.T

    xr = xyz[..., 0] / _D65_XN
    yr = xyz[..., 1] / _D65_YN
    zr = xyz[..., 2] / _D65_ZN

    fx, fy, fz = _f(xr), _f(yr), _f(zr)

    L = 116.0 * fy - 16.0
    a = 500.0 * (fx - fy)
    b = 200.0 * (fy - fz)
    return np.stack([L, a, b], axis=-1)


def ita_degrees(L: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Individual Typology Angle, in degrees: ``arctan((L*-50)/b*) * 180/pi``.

    Plain ``arctan`` of the ratio - the literature formula (Kolm/Chardon,
    Groh et al.) - not ``arctan2`` of the two components: ``arctan2`` is
    quadrant-aware and would wrap negative-``b*`` readings into (90, 180],
    which contradicts the "ITA cannot leave [-90, 90]" invariant that
    design.md's error table asserts "by arctan construction".
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.degrees(np.arctan((np.asarray(L) - 50.0) / np.asarray(b)))


def fst_bucket(ita: float) -> int:
    """Map an aggregate ITA (degrees) to a Fitzpatrick bucket (1-6).

    ``FST_THRESHOLDS`` are the Groh et al. empirical cut points, descending.
    A value exactly ON a threshold belongs to the DARKER (higher-numbered)
    bucket - i.e. thresholds are exclusive lower bounds for the lighter
    bucket, matching the ``>`` comparisons in the original ITA-to-FST table.
    """
    bucket = len(FST_THRESHOLDS) + 1
    for i, threshold in enumerate(FST_THRESHOLDS, start=1):
        if ita > threshold:
            bucket = i
            break
    return bucket


@dataclass
class FSTEstimate:
    """Result of one FST estimation call. Never dropped silently (spec.md:
    "Explicit Bucket-or-Unavailable Output") - always either a bucket or an
    explicit ``status`` explaining why one is not available."""

    bucket: int | None  # 1..6, None when unavailable
    ita: float  # NaN when unavailable
    skin_fraction: float
    status: str  # "ok" | "no_skin" | "degenerate_mask" | "ita_out_of_range"


def _is_degenerate(mask: np.ndarray) -> bool:
    """True when ``mask``'s area is outside the plausible single-lesion
    range (1%-90%) - too small to be a real segmentation, or so large it
    likely captured the whole frame instead of one lesion."""
    area = float(mask.mean())
    return not (MIN_MASK_AREA_FRACTION <= area <= MAX_MASK_AREA_FRACTION)


def estimate_fst(img: np.ndarray, mask: np.ndarray | None = None) -> FSTEstimate:
    """Mask-guided FST estimate for one sRGB image, float (H, W, 3) in [0, 1].

    ``mask`` is the primary lesion segmentation (bool (H, W), True = lesion -
    e.g. a U-Net's output in Phase 3). When omitted, or when it is
    degenerate (area outside 1%-90%), the pure-numpy Otsu fallback from
    ``pretrain.lesion_guided`` is used instead (D3: two-tier masking, one
    contract - both return bool (H, W), skin = ~mask).

    A non-bool ``mask`` (e.g. an int/uint8 0/1 array) is coerced to bool:
    downstream code relies on ``~mask`` being logical negation, which on an
    integer dtype would instead be bitwise NOT on the raw integers (wrong
    fancy-indexing into ``lab[skin]``, silently, no error raised).
    """
    if mask is not None:
        mask = np.asarray(mask, dtype=bool)

    gray = img.mean(axis=-1)
    border = gray < BORDER_LUMINANCE_THRESHOLD

    if mask is None or _is_degenerate(mask):
        mask = lesion_guided.lesion_mask(gray)

    if _is_degenerate(mask):
        skin = ~mask & ~border
        return FSTEstimate(
            bucket=None,
            ita=float("nan"),
            skin_fraction=float(skin.mean()),
            status="degenerate_mask",
        )

    skin = ~mask & ~border
    skin_fraction = float(skin.mean())

    if skin_fraction < MIN_SKIN_FRACTION:
        return FSTEstimate(
            bucket=None, ita=float("nan"), skin_fraction=skin_fraction, status="no_skin"
        )

    lab = srgb_to_lab(img)
    skin_lab = lab[skin]
    L_med = np.median(skin_lab[:, 0])
    b_med = np.median(skin_lab[:, 2])
    ita = float(ita_degrees(np.asarray(L_med), np.asarray(b_med)))

    # Finite ITA cannot leave [-90, 90] by arctan construction (see
    # ita_degrees) - but an achromatic skin median at L*=50 makes the
    # (L*-50)/b* ratio a literal 0/0, which is NaN, and NaN comparisons
    # are always False. Check explicitly (never a bare assert, which would
    # both crash the whole batch on one atypical image - the opposite of
    # lesion_guided.py's philosophy - and be silently skipped entirely
    # under python -O / PYTHONOPTIMIZE=1) and degrade gracefully, matching
    # every other "no result" branch in this function and the contract
    # documented on FSTEstimate.status.
    if not (-90.0 <= ita <= 90.0):
        return FSTEstimate(
            bucket=None, ita=ita, skin_fraction=skin_fraction, status="ita_out_of_range"
        )

    return FSTEstimate(
        bucket=fst_bucket(ita), ita=ita, skin_fraction=skin_fraction, status="ok"
    )


# ============================================================================
# gates.py  --  inlined from src/dermssl/gates.py (do not edit here)
# ============================================================================

"""Acceptance gates: the explicit pass/fail contract of a run.

Why this module exists
----------------------
This logic used to live inside the notebook, and that is exactly why it shipped
with a bug: the self-supervised quality gate averaged the SSL backbone together
with the supervised baseline it is supposed to beat, so a run where DINOv2 scored
0.8005 was reported as 0.794 and failed a 0.80 threshold it had actually cleared.
Untested code in a notebook cell is still untested code. Moving the gates here
puts them under ``tests/``.

Three states, not two
---------------------
``pass`` / ``fail`` / ``skip``. The third one matters: in ``SMOKE`` mode the
notebook states up front that the numbers are not valid results, so enforcing a
*quality* threshold on them is incoherent. Quality gates are skipped in smoke
runs and enforced in real ones. Pipeline gates (does it run at all, is the split
clean, is the backbone frozen) are enforced always - those are exactly what a
smoke run is for.
"""


from dataclasses import dataclass
from typing import Iterable, Sequence

PASS = "pass"
FAIL = "fail"
SKIP = "skip"

DEFAULT_KNN_AUC_THRESHOLD = 0.80
SSL_PREFIX = "ssl_"


@dataclass
class Gate:
    """One acceptance criterion and its outcome."""

    name: str
    criterion: str
    value: object
    status: str

    @property
    def passed(self) -> bool:
        return self.status == PASS

    @property
    def blocking(self) -> bool:
        """A skipped gate does not block; a failed one does."""
        return self.status == FAIL

    def render(self) -> str:
        label = {PASS: "PASA", FAIL: "FALLA", SKIP: "OMITIDA"}[self.status]
        return f"[{label}] {self.name}: {self.value}  ({self.criterion})"

    def as_dict(self) -> dict:
        return {
            "gate": self.name,
            "criterio": self.criterion,
            "valor": self.value,
            "estado": self.status,
            "pasa": self.passed,
        }


def _decide(ok: bool, *, skip: bool = False) -> str:
    if skip:
        return SKIP
    return PASS if ok else FAIL


def ssl_knn_auc(
    rows: Sequence[dict],
    fraction: float | None = None,
    probe: str = "knn",
    ssl_prefix: str = SSL_PREFIX,
) -> float:
    """Mean k-NN AUC of the **self-supervised backbones only**.

    The bug this function replaces averaged every backbone together. Mixing the
    supervised baseline into the self-supervised score makes the metric answer a
    question nobody asked, and drags it toward the baseline.

    ``fraction=None`` means "the largest label fraction present". Rows with a NaN
    AUC (an undefined class, a failed probe) are ignored rather than propagated.
    """
    candidates = [
        r
        for r in rows
        if str(r.get("backbone", "")).startswith(ssl_prefix)
        and r.get("probe") == probe
        and r.get("auc") == r.get("auc")  # drop NaN
    ]
    if not candidates:
        return float("nan")
    target = max(r["fraction"] for r in candidates) if fraction is None else fraction
    selected = [r["auc"] for r in candidates if r["fraction"] == target]
    if not selected:
        return float("nan")
    return float(sum(selected) / len(selected))


def has_architecture_control(backbones: Iterable[str], ssl_prefix: str = SSL_PREFIX) -> bool:
    """True when a supervised backbone shares the family of an SSL one.

    Without it, a win by DINOv2 (a ViT) over ResNet-50 (a CNN) cannot be
    attributed to self-supervision rather than to the architecture. The check is
    name based: an SSL ViT needs a supervised ViT alongside it.
    """
    names = list(backbones)
    ssl_vits = [n for n in names if n.startswith(ssl_prefix) and "vit" in n.lower()]
    sup_vits = [n for n in names if not n.startswith(ssl_prefix) and "vit" in n.lower()]
    if ssl_vits:
        return bool(sup_vits)
    return True  # no SSL ViT in the run: nothing to control for


def evaluate_gates(
    *,
    backend: str,
    ham_images: int,
    lesion_overlap: int,
    rows_total: int,
    rows_ok: int,
    frozen_verified: bool,
    backbones: Sequence[str],
    grid_rows: Sequence[dict],
    smoke: bool,
    knn_auc_threshold: float = DEFAULT_KNN_AUC_THRESHOLD,
) -> list[Gate]:
    """Build the full gate report for a run.

    Pipeline gates are enforced in every mode. Quality gates are skipped under
    ``smoke`` because a smoke run deliberately uses a tiny subset.
    """
    auc = ssl_knn_auc(grid_rows)
    gates = [
        Gate(
            "G0 resolucion de datos",
            "backend resuelto y HAM10000 localizado",
            f"{backend} / {ham_images} imgs",
            _decide(bool(backend) and ham_images > 0),
        ),
        Gate(
            "G0 anti-leakage",
            "0 lesiones compartidas train/val",
            lesion_overlap,
            _decide(lesion_overlap == 0),
        ),
        Gate(
            "G1 pipeline end-to-end",
            "toda la rejilla corre sin fallos",
            f"{rows_ok}/{rows_total}",
            _decide(rows_total > 0 and rows_ok == rows_total),
        ),
        Gate(
            "G1 backbone congelado",
            "0 parametros entrenables en todo backbone usado",
            "verificado por assert en la extraccion" if frozen_verified else "NO verificado",
            _decide(frozen_verified),
        ),
        Gate(
            "G2 calidad SSL sin entrenar nada",
            f"AUC del probe kNN del backbone SSL > {knn_auc_threshold}",
            round(auc, 4) if auc == auc else "n/a",
            _decide(auc == auc and auc > knn_auc_threshold, skip=smoke),
        ),
        Gate(
            "G2 control de arquitectura",
            "cada ViT auto-supervisado tiene un ViT supervisado al lado",
            [b for b in backbones if not b.startswith(SSL_PREFIX)] or "ninguno",
            _decide(has_architecture_control(backbones), skip=smoke),
        ),
    ]
    return gates


FAIRNESS_METRIC = "macro_f1"
DEFAULT_FAIRNESS_MAX_GAP = 0.15
DEFAULT_FAIRNESS_MIN_BUCKET_N = 30


def fairness_gate(
    summary: dict,
    *,
    max_gap: float = DEFAULT_FAIRNESS_MAX_GAP,
    min_bucket_n: int = DEFAULT_FAIRNESS_MIN_BUCKET_N,
) -> Gate:
    """G3: how much macro-F1 varies across Fitzpatrick skin-type buckets.

    Standalone by design (D1 in design.md): composes over ``evaluate_gates``
    instead of extending it, so this module stays stdlib-only and its
    signature never changes. Caller concatenates::

        gates = evaluate_gates(...) + [fairness_gate(strat)]

    ``summary`` is the output of ``metrics.stratified_summary`` - this
    function never touches FST/ITA math, only the numbers already computed.
    Mirrors the G2 smoke-skip precedent: fewer than two buckets with at least
    ``min_bucket_n`` samples means there is nothing trustworthy to compare, so
    the gate SKIPs - it never FAILs on thin or absent data (spec.md,
    fairness-evaluation req 3). ``max_gap=0.15`` is the reference target from
    docs/07 and docs/11 ("Brecha de fairness (Fitzpatrick) <= 0.15") - a
    secondary KPI, not the thesis's primary acceptance criterion.
    """
    qualifying = {
        bucket: entry[FAIRNESS_METRIC]
        for bucket, entry in summary.items()
        if entry.get("status") == "OK" and entry.get("count", 0) >= min_bucket_n
    }
    criterion = f"brecha de {FAIRNESS_METRIC} entre fototipos <= {max_gap}"
    if len(qualifying) < 2:
        omitted = sorted(str(bucket) for bucket in summary if bucket not in qualifying)
        return Gate(
            "G3 equidad por fototipo",
            criterion,
            f"menos de 2 fototipos con >= {min_bucket_n} muestras "
            f"(omitidos: {', '.join(omitted) if omitted else 'sin datos'})",
            SKIP,
        )
    gap = max(qualifying.values()) - min(qualifying.values())
    return Gate("G3 equidad por fototipo", criterion, round(gap, 4), _decide(gap <= max_gap))


def report(gates: Sequence[Gate]) -> str:
    """Human-readable block, plus an unambiguous verdict line."""
    lines = ["=== REPORTE DE COMPUERTAS ==="]
    lines.extend(f"  {g.render()}" for g in gates)
    blocking = [g for g in gates if g.blocking]
    skipped = [g for g in gates if g.status == SKIP]
    if blocking:
        lines.append("")
        lines.append(f"HAY {len(blocking)} COMPUERTA(S) EN FALLA - revisar arriba")
    else:
        note = f" ({len(skipped)} omitida(s) por SMOKE)" if skipped else ""
        lines.append("")
        lines.append(f"TODAS LAS COMPUERTAS EXIGIBLES PASAN{note}")
    return "\n".join(lines)


def all_blocking_pass(gates: Sequence[Gate]) -> bool:
    """True when nothing failed. Skipped gates do not block."""
    return not any(g.blocking for g in gates)


# ============================================================================
# circle.py  --  inlined from src/dermssl/circle.py (do not edit here)
# ============================================================================

"""Stage 2 mitigation: CIRCLe-style LAB color transformer, FST-shift sampling,
and synthetic-blend policy, replicated at the trainable-probe level only
(Pakzad et al., arXiv:2208.13528; Corbin & Marques, IEEE Access 2023).

Why this module exists, and what it is NOT
--------------------------------------------
Per proposal.md, Stage 2 is "gated on Stage 1 showing measurable bias" - no real
Stage 1 Kaggle run has happened yet (PR3's ``scripts/kaggle_fst_manifest.py`` is
still a manual step pending a GPU session), so there is no evidence yet that
bias mitigation is even needed. This module is Stage 2's MACHINERY: a
deterministic color transform, a bounded FST-shift sampler, and a blend
policy - built and tested ahead of time so it is ready to use the moment a real
``fst_manifest.csv`` exists. Nothing here measures or claims a real fairness
improvement; :func:`should_run_stage2` is the explicit stage-gate that keeps
this module from running against real data until Stage 1 says it is worth it
(spec.md, circle-adaptation, "Stage-Gate Dependency on Stage 1").

Pure numpy, no scikit-image at runtime - same reasoning as fitzpatrick.py's D2
in design.md: ``src/dermssl/__init__.py::MODULE_ORDER`` modules are inlined and
executed by the notebook-build test, so a module-level ``import skimage`` here
would break both that test and the Kaggle cell.
"""


import numpy as np

from dermssl.fitzpatrick import (
    FST_THRESHOLDS,
    MAX_MASK_AREA_FRACTION,
    MIN_MASK_AREA_FRACTION,
    srgb_to_lab,
)
from dermssl.gates import FAIL

# Maximum FST-bucket shift a synthetic sample may request (spec.md req 4:
# "MUST NOT allow a synthetic shift of more than 2 FST levels from the source
# image's estimated bucket").
MAX_FST_SHIFT = 2

# L* units shifted per FST-bucket level. A heuristic magnitude, not a measured
# one: darker target buckets (higher bucket number) move L* down. Chosen so a
# 1-bucket shift is visually apparent without pushing L* out of [0, 100] for a
# 2-bucket shift on a typical skin-tone swatch.
FST_SHIFT_L_STEP = 8.0

# D65 illuminant, 2-degree observer - MUST match fitzpatrick.py's private
# ``_D65_X/Y/ZN`` and ``_SRGB_TO_XYZ`` bit-for-bit, or lab_to_srgb stops being
# the inverse of fitzpatrick.srgb_to_lab (tests/test_circle.py's round-trip
# test pins this down). Duplicated rather than imported because fitzpatrick.py
# keeps those constants private (leading underscore) by design.
_D65_XN = 0.95047
_D65_YN = 1.0
_D65_ZN = 1.08883

_SRGB_TO_XYZ = np.array(
    [
        [0.412453, 0.357580, 0.180423],
        [0.212671, 0.715160, 0.072169],
        [0.019334, 0.119193, 0.950227],
    ]
)
_XYZ_TO_SRGB = np.linalg.inv(_SRGB_TO_XYZ)


def _f_inv(t: np.ndarray) -> np.ndarray:
    """Inverse of fitzpatrick.py's ``_f`` (CIE Lab nonlinearity), same
    piecewise breakpoint (0.008856) so the round trip matches skimage's
    ``lab2xyz`` convention within the same tolerance ``srgb_to_lab`` holds."""
    cube = t**3
    linear = (t - 16.0 / 116.0) / 7.787
    return np.where(cube > 0.008856, cube, linear)


def _gamma_encode(linear: np.ndarray) -> np.ndarray:
    """Linear-light [0, 1] -> sRGB companded [0, 1]. Inverse of fitzpatrick's
    ``_inverse_gamma``."""
    low = linear * 12.92
    high = 1.055 * np.power(np.clip(linear, 1e-12, None), 1.0 / 2.4) - 0.055
    return np.where(linear <= 0.0031308, low, high)


def lab_to_srgb(lab: np.ndarray) -> np.ndarray:
    """Convert CIE Lab back to sRGB, float in [0, 1], shape (..., 3).

    Inverse of ``fitzpatrick.srgb_to_lab`` (D65, 2-degree observer). Output is
    clipped to a valid [0, 1] range: an arbitrary Lab triple is not always a
    representable sRGB color, and silently returning out-of-gamut values would
    violate the "valid LAB/RGB range" contract this module is tested against.
    """
    lab = np.asarray(lab, dtype=float)
    L, a, b = lab[..., 0], lab[..., 1], lab[..., 2]
    fy = (L + 16.0) / 116.0
    fx = fy + a / 500.0
    fz = fy - b / 200.0

    xr, yr, zr = _f_inv(fx), _f_inv(fy), _f_inv(fz)
    xyz = np.stack([xr * _D65_XN, yr * _D65_YN, zr * _D65_ZN], axis=-1)

    linear = xyz @ _XYZ_TO_SRGB.T
    linear = np.clip(linear, 0.0, None)  # guard tiny negative float noise
    srgb = _gamma_encode(linear)
    return np.clip(srgb, 0.0, 1.0)


def clamp_fst_shift(requested_shift: int, *, max_shift: int = MAX_FST_SHIFT) -> int:
    """Clamp a requested FST-bucket shift to ``[-max_shift, max_shift]``.

    spec.md req 4: "MUST clamp the shift to the maximum allowed ... and MUST
    NOT silently apply the larger shift" - this is the explicit clamp, never a
    silent pass-through of an out-of-bound request.
    """
    if requested_shift > max_shift:
        return max_shift
    if requested_shift < -max_shift:
        return -max_shift
    return requested_shift


def target_fst_bucket(
    source_bucket: int, requested_shift: int, *, max_shift: int = MAX_FST_SHIFT
) -> int:
    """Resulting FST bucket (1..6) after a possibly-clamped shift.

    spec.md scenario: source bucket 2, requested shift 2 -> target bucket 4.
    A requested shift beyond ``max_shift`` clamps first (see
    :func:`clamp_fst_shift`), then the result is bounded to the valid bucket
    range so a shift near the ends of the scale cannot produce bucket 0 or 7.
    """
    shift = clamp_fst_shift(requested_shift, max_shift=max_shift)
    n_buckets = len(FST_THRESHOLDS) + 1
    return int(min(max(source_bucket + shift, 1), n_buckets))


def _is_degenerate_mask(mask: np.ndarray) -> bool:
    """Same bounds as fitzpatrick.py's private ``_is_degenerate`` (D3: one
    mask contract, two producers)."""
    area = float(mask.mean())
    return not (MIN_MASK_AREA_FRACTION <= area <= MAX_MASK_AREA_FRACTION)


def shift_skin_tone(
    img: np.ndarray, mask: np.ndarray | None, shift_buckets: int, *, max_shift: int = MAX_FST_SHIFT
) -> np.ndarray:
    """Deterministic LAB color transformer: shifts skin-pixel L* by
    ``shift_buckets`` FST-bucket levels; lesion pixels (``mask`` True) are
    passed through pixel-for-pixel unchanged.

    ``img`` is sRGB float (H, W, 3) in [0, 1]. ``mask`` is the same contract
    as ``fitzpatrick.estimate_fst``: bool (H, W), True = lesion; a non-bool
    mask is coerced (mirrors fitzpatrick's own dtype guard so ``~mask`` stays
    logical negation, not bitwise NOT on raw integers).

    Raises ``ValueError`` when the mask is missing or degenerate (area outside
    1%-90%) instead of silently transforming the whole frame - spec.md req 2:
    "MUST NOT silently apply an unprotected color shift over the lesion".
    """
    if mask is None:
        raise ValueError("shift_skin_tone requires an explicit lesion mask")
    mask = np.asarray(mask, dtype=bool)
    if _is_degenerate_mask(mask):
        raise ValueError(
            f"degenerate lesion mask (area={float(mask.mean()):.4f}); "
            "cannot protect the lesion region"
        )

    shift = clamp_fst_shift(shift_buckets, max_shift=max_shift)
    img = np.asarray(img, dtype=float)
    lab = srgb_to_lab(img)

    shifted_L = np.clip(lab[..., 0] - shift * FST_SHIFT_L_STEP, 0.0, 100.0)
    shifted_lab = lab.copy()
    shifted_lab[..., 0] = np.where(mask, lab[..., 0], shifted_L)

    out = lab_to_srgb(shifted_lab)
    out = np.where(mask[..., None], img, out)  # exact passthrough over the lesion
    return np.clip(out, 0.0, 1.0)


def synthetic_blend_mask(
    n_samples: int, *, target_ratio: float = 0.5, seed: int = 42
) -> np.ndarray:
    """Boolean mask (length ``n_samples``) selecting which samples get an
    FST-shifted synthetic counterpart, sized so ``target_ratio`` of the batch
    is synthetic (spec.md req 4, default ~50/50). Deterministic for a given
    ``seed``.
    """
    if not (0.0 <= target_ratio <= 1.0):
        raise ValueError(f"target_ratio must be within [0, 1], got {target_ratio!r}")

    n_synthetic = int(round(n_samples * target_ratio))
    mask = np.zeros(n_samples, dtype=bool)
    if n_synthetic == 0 or n_samples == 0:
        return mask
    rng = np.random.RandomState(seed)
    idx = rng.choice(n_samples, size=n_synthetic, replace=False)
    mask[idx] = True
    return mask


def should_run_stage2(gate_status: str) -> bool:
    """Stage-gate dependency on Stage 1 (spec.md req 1): Stage 2 is only
    required to run when Stage 1's fairness gate reports ``FAIL``. It is
    skippable - not forced - when Stage 1 reports ``PASS`` or ``SKIP``.
    """
    return gate_status == FAIL


print("nucleo dermssl inlined:", ['patch_guard', 'data_resolver', 'label_regimes', 'metrics', 'probes', 'feature_cache', 'fitzpatrick', 'gates', 'circle'])

## 1. Localizar los datos (el cambio que permite correr fuera de Kaggle)

v4 tenia `/kaggle/input` escrito a mano, asi que solo funcionaba dentro de Kaggle.
`resolve_datasets()` prueba tres estrategias y avisa cual uso:

- **kaggle-input**: los datasets que Kaggle ya monto. Instantaneo, sin internet.
- **kagglehub**: los baja con el paquete `kagglehub` usando tus credenciales. Sirve en
  cualquier maquina, incluido el cluster de la universidad.
- **local**: copias ya guardadas en un disco (scratch, NFS).

El mismo notebook, sin editar una linea, corre en los tres sitios.

> **Seguridad**: las credenciales van en variables de entorno (`KAGGLE_USERNAME`,
> `KAGGLE_KEY`) o en `~/.kaggle/kaggle.json` con permisos 600. **Nunca** dentro del
> notebook ni en el repositorio.

In [ ]:
# index_pool=False: el pool sin etiquetas NO se usa con el backbone congelado, y
# recorrer las ~48k imagenes de ISIC costo 157 de los 192 s de la corrida del 29-07.
# Se activa (index_pool=True) recien en el pre-entrenamiento continuo.
data = resolve_datasets(backend=DATA_BACKEND, local_root=LOCAL_ROOT, index_pool=False)
print(json.dumps(data.summary(), indent=2))

meta = pd.read_csv(data.ham_metadata_csv)
meta = meta[meta.image_id.isin(data.ham_images)].reset_index(drop=True)
classes = sorted(meta.dx.unique())
cls2idx = {c: i for i, c in enumerate(classes)}
id2label = {r.image_id: cls2idx[r.dx] for r in meta.itertuples()}
N_CLASSES = len(classes)
print("\nclases:", classes)
print(meta.dx.value_counts().to_string())

## 2. Particion honesta por lesion (anti-leakage)

HAM10000 contiene **varias fotos de la misma lesion**. Si esas fotos se reparten entre
entrenamiento y validacion, el modelo ya vio la respuesta y la metrica sube sin que el
modelo haya mejorado: eso es *data leakage*. Fue el pecado de la v1 (0.873 falso).

La particion se hace por `lesion_id` con `StratifiedGroupKFold`, y un `assert` verifica
**cero** lesiones compartidas. Si ese assert falla, el notebook se detiene: preferimos
parar que publicar un numero inflado.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

all_ids = meta.image_id.tolist()
y_all   = np.array([id2label[i] for i in all_ids])
groups  = meta.set_index("image_id").loc[all_ids, "lesion_id"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = next(sgkf.split(all_ids, y_all, groups))
tr_ids = [all_ids[i] for i in tr_idx]
va_ids = [all_ids[i] for i in va_idx]

lookup  = meta.set_index("image_id")
overlap = set(lookup.loc[tr_ids, "lesion_id"]) & set(lookup.loc[va_ids, "lesion_id"])
assert len(overlap) == 0, f"LEAKAGE: {len(overlap)} lesiones en train y val a la vez"

if SMOKE:  # subconjunto estratificado para que el humo corra en minutos
    y_tr_full = np.array([id2label[i] for i in tr_ids])
    y_va_full = np.array([id2label[i] for i in va_ids])
    tr_ids = [tr_ids[i] for i in stratified_regime_indices(y_tr_full, 0.08, seed=42, min_per_class=12)]
    va_ids = [va_ids[i] for i in stratified_regime_indices(y_va_full, 0.08, seed=42, min_per_class=6)]

y_train = np.array([id2label[i] for i in tr_ids])
y_val   = np.array([id2label[i] for i in va_ids])
print(f"train={len(tr_ids)} | val={len(va_ids)} | lesiones compartidas={len(overlap)} (debe ser 0)")
print("regimenes de etiquetas disponibles:")
for row in describe_regimes(y_train, build_regimes(y_train, FRACTIONS, seed=SEEDS[0])):
    print(f"  {row['fraction']:>5.0%} -> {row['n_labels']:>5} etiquetas "
          f"({row['pct_of_train']:>5.1f}% del train) | clase mas chica: {row['min_class_count']}")

## 3. Backbones congelados (y el control que hace justa la comparacion)

Tres modelos, ninguno entrenado aqui:

| Clave | Que es | Etiquetas usadas para crearlo |
|---|---|---|
| `ssl_dinov2_vits14` | DINOv2 ViT-S/14 | **ninguna** (LVD-142M, auto-supervisado) |
| `sup_vit_b16` | ViT-B/16 de torchvision | 1.28 M (ImageNet) |
| `sup_resnet50` | ResNet-50 de torchvision | 1.28 M (ImageNet) |

**Por que `sup_vit_b16` es obligatorio.** Si comparamos DINOv2 (un ViT) contra ResNet-50 y
DINOv2 gana, no sabemos si gano *el metodo auto-supervisado* o simplemente *la
arquitectura ViT*. `sup_vit_b16` es un ViT supervisado: aisla esa variable. Sin este
control la conclusion no se sostiene ante un jurado.

**La trampa del parche 14.** Un ViT corta la imagen en cuadraditos. DINOv2 usa 14x14, asi
que el lado de la imagen debe ser multiplo de 14. `224 = 16 x 14` sirve, pero **256 no**
(256/14 = 18.28...). Las imagenes de ISIC 2020 vienen a 256. `resolve_resolution()`
detecta el caso y ajusta, imprimiendo el cambio para que nunca sea invisible.

In [ ]:
from torchvision.models import (
    AlexNet_Weights, EfficientNet_B0_Weights, ResNet50_Weights, VGG19_Weights,
    ViT_B_16_Weights, alexnet, efficientnet_b0, resnet50, vgg19, vit_b_16)

BACKBONE_SPECS = {
    "ssl_dinov2_vits14": {"patch_name": "dinov2_vits14", "labels_used": 0,
                          "family": "vit", "pretrain": "DINOv2 LVD-142M (sin etiquetas)"},
    "ssl_dinov2_vitb14": {"patch_name": "dinov2_vitb14", "labels_used": 0,
                          "family": "vit", "pretrain": "DINOv2 LVD-142M (sin etiquetas)"},
    "sup_vit_b16":       {"patch_name": "vit_base_patch16_224", "labels_used": 1_281_167,
                          "family": "vit", "pretrain": "ImageNet-1k supervisado"},
    "sup_resnet50":      {"patch_name": "resnet50", "labels_used": 1_281_167,
                          "family": "cnn", "pretrain": "ImageNet-1k supervisado"},
    "ssl_simsiam_r50":   {"patch_name": "resnet50", "labels_used": 1_281_167,
                          "family": "cnn", "pretrain": "SimSiam sobre init ImageNet (v4)"},
    "sup_alexnet":       {"patch_name": "alexnet", "labels_used": 1_281_167,
                          "family": "cnn", "pretrain": "ImageNet-1k supervisado (variante 2014)"},
    "sup_vgg19":         {"patch_name": "vgg19", "labels_used": 1_281_167,
                          "family": "cnn", "pretrain": "ImageNet-1k supervisado"},
    "sup_effnet_b0":     {"patch_name": "efficientnet_b0", "labels_used": 1_281_167,
                          "family": "cnn", "pretrain": "ImageNet-1k supervisado (AutoAugment)"},
}

# Punto de extraccion unico para CNNs sin avgpool+fc integrados: GAP del ultimo
# bloque convolucional (mismo protocolo que resnet50 con fc=Identity). Mezclar
# fc7 (4096-d) en unas y pooling en otras compararia protocolos, no modelos.
def _gap_wrapper(features_module):
    return nn.Sequential(features_module, nn.AdaptiveAvgPool2d(1), nn.Flatten(1))

def _load_dinov2(name, res):
    """Tres rutas, en orden de preferencia. Devuelve (modelo, como_se_cargo)."""
    errors = []
    size = "small" if "vits" in name else "base"
    try:  # 1) timm: preinstalado en Kaggle, pesos desde Hugging Face
        import timm
        model = timm.create_model(f"vit_{size}_patch14_dinov2.lvd142m",
                                  pretrained=True, num_classes=0, img_size=res)
        return model, "timm/lvd142m"
    except Exception as exc:
        errors.append(f"timm: {exc}")
    try:  # 2) torch.hub oficial de facebookresearch
        model = torch.hub.load("facebookresearch/dinov2", name, trust_repo=True)
        return model, "torch.hub/facebookresearch"
    except Exception as exc:
        errors.append(f"torch.hub: {exc}")
    local = list(Path("/kaggle/input").glob(f"**/{name}*.pth")) if Path("/kaggle/input").is_dir() else []
    if local:  # 3) checkpoint subido como Input (para sesiones sin internet)
        import timm
        model = timm.create_model(f"vit_{size}_patch14_dinov2", pretrained=False,
                                  num_classes=0, img_size=res)
        model.load_state_dict(torch.load(local[0], map_location="cpu"), strict=False)
        return model, f"local:{local[0].name}"
    raise RuntimeError(
        "No se pudo cargar DINOv2. Activa Internet en Settings, o sube los pesos como "
        "Input dataset. Intentos:\n  - " + "\n  - ".join(errors))

def build_frozen_backbone(key, res):
    """Devuelve (modelo_en_eval, dim_features, fuente). NUNCA entrena."""
    if key.startswith("ssl_dinov2"):
        model, source = _load_dinov2(BACKBONE_SPECS[key]["patch_name"], res)
    elif key == "sup_vit_b16":
        model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        model.heads = nn.Identity()          # se queda el token CLS crudo
        source = "torchvision/IMAGENET1K_V1"
    elif key in ("sup_resnet50", "ssl_simsiam_r50"):
        if key == "ssl_simsiam_r50":
            ckpts = list(Path("/kaggle/input").glob("**/backbone_domain_adaptive_r50.pth"))
            if not ckpts:
                raise FileNotFoundError("falta backbone_domain_adaptive_r50.pth (salida de v4)")
            model = resnet50(weights=None)
            model.load_state_dict(torch.load(ckpts[0], map_location="cpu"), strict=False)
            source = f"v4:{ckpts[0].name}"
        else:
            model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
            source = "torchvision/IMAGENET1K_V2"
        model.fc = nn.Identity()
    elif key == "sup_alexnet":
        model = _gap_wrapper(alexnet(weights=AlexNet_Weights.IMAGENET1K_V1).features)
        source = "torchvision/IMAGENET1K_V1 (One-weird-trick 2014)"
    elif key == "sup_vgg19":
        model = _gap_wrapper(vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features)
        source = "torchvision/IMAGENET1K_V1 (sin BN, como el paper)"
    elif key == "sup_effnet_b0":
        model = _gap_wrapper(efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).features)
        source = "torchvision/IMAGENET1K_V1"
    else:
        raise ValueError(f"backbone desconocido: {key}")

    for p in model.parameters():
        p.requires_grad = False              # congelado, de forma explicita y verificable
    model.eval().to(device)
    with torch.no_grad():
        dim = model(torch.zeros(1, 3, res, res, device=device)).shape[-1]
    return model, int(dim), source

# Resolucion por backbone: se ajusta al tamano de parche y se reporta el cambio.
# torchvision/vit_b_16 lleva la resolucion incrustada en sus pesos: solo acepta 224.
FIXED_RES = {"sup_vit_b16": 224}
RES_BY_BACKBONE = {}
for key in BACKBONES:
    if key in FIXED_RES:
        res, note = FIXED_RES[key], f"fija en {FIXED_RES[key]} (pesos de torchvision)"
    else:
        res, note = resolve_resolution(BASE_RES, BACKBONE_SPECS[key]["patch_name"])
    RES_BY_BACKBONE[key] = res
    print(f"{key:<20} res={res:<4} patch={patch_size_for(BACKBONE_SPECS[key]['patch_name'])} | {note}")
print(f"\n[control de la trampa] 256 con parche 14 -> {resolve_resolution(256, 'dinov2_vitb14')[1]}")

## 4. Extraer features una sola vez y cachearlas

Pasar 10 000 imagenes por un ViT cuesta minutos de GPU. Ajustar un probe sobre los
vectores resultantes cuesta milisegundos. Entonces: se extrae **una vez**, se guarda a
disco, y despues se prueban todos los probes, regimenes y semillas que queramos, gratis.

Eso convierte "3 backbones x 3 probes x 6 regimenes x 3 semillas" (162 experimentos) en
minutos en vez de dias. Y hace el notebook resistente a caidas: si la sesion muere
despues de extraer, al reiniciar reusa lo cacheado.

La clave del cache incluye una huella del **orden exacto** de los ids. Si cambia la
particion, cambia la clave, y el cache viejo se ignora en vez de contaminar los
resultados en silencio.

In [ ]:
cache = FeatureCache(CACHE_DIR)

# EfficientNet-B0 fue entrenado con remuestreo BICUBIC; evaluarlo con BILINEAR
# lo saca de su distribucion (regla: preprocesamiento por modelo, docs/17).
INTERP_BY_BACKBONE = {"sup_effnet_b0": transforms.InterpolationMode.BICUBIC}

def eval_transform(res, interp=transforms.InterpolationMode.BILINEAR):
    """Sin aumentos ni aleatoriedad: la extraccion debe ser reproducible."""
    return transforms.Compose([
        transforms.Resize(int(res * 1.14), interpolation=interp),
        transforms.CenterCrop(res),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

def load_rgb(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception as exc:
        print(f"[WARN] imagen ilegible {path}: {exc} (placeholder negro)")
        return Image.new("RGB", (224, 224), (0, 0, 0))

class ImageIdDataset(Dataset):
    def __init__(self, ids, tf):
        self.ids, self.tf = ids, tf
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        image_id = self.ids[i]
        return self.tf(load_rgb(data.ham_images[image_id])), id2label[image_id]

@torch.no_grad()
def extract(model, ids, res, interp=transforms.InterpolationMode.BILINEAR):
    dl = DataLoader(ImageIdDataset(ids, eval_transform(res, interp)), batch_size=BATCH,
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    feats, labels = [], []
    for images, batch_labels in dl:
        images = images.to(device, non_blocking=True)
        if AMP:
            with torch.amp.autocast("cuda"):
                out = model(images)
        else:
            out = model(images)
        feats.append(out.float().cpu().numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(feats), np.concatenate(labels)

FEATURES, EXTRACTION_LOG = {}, []
for key in BACKBONES:
    res = RES_BY_BACKBONE[key]
    interp = INTERP_BY_BACKBONE.get(key, transforms.InterpolationMode.BILINEAR)
    keys = {s: cache_key(key, s, res, ids) for s, ids in (("train", tr_ids), ("val", va_ids))}
    need_gpu = not all(cache.has(k) for k in keys.values())

    model = source = None
    if need_gpu:
        model, dim, source = build_frozen_backbone(key, res)
        frozen = sum(p.requires_grad for p in model.parameters())
        assert frozen == 0, f"{key}: {frozen} parametros entrenables - el backbone NO esta congelado"
        print(f"\n[{key}] cargado de {source} | dim={dim} | res={res} | parametros entrenables=0")

    per_split, wall = {}, 0.0
    for split, ids in (("train", tr_ids), ("val", va_ids)):
        t0 = time.time()
        entry = cache.get_or_extract(
            keys[split], lambda ids=ids, res=res, interp=interp: extract(model, ids, res, interp),
            meta={"backbone": key, "split": split, "resolution": res, "source": source},
        )
        took = time.time() - t0
        wall += took
        per_split[split] = entry
        print(f"  {split:<5} n={len(entry.features):<5} dim={entry.features.shape[1]:<5} "
              f"{'extraido' if need_gpu else 'cacheado'} en {took:.1f}s")

    FEATURES[key] = {"train": (per_split["train"].features, per_split["train"].labels),
                     "val":   (per_split["val"].features,   per_split["val"].labels)}
    EXTRACTION_LOG.append({
        "backbone": key, "resolution": res, "source": source,
        "dim": int(per_split["val"].features.shape[1]),
        "images": len(tr_ids) + len(va_ids),
        "wall_seconds": round(wall, 2),
        "images_per_second": round((len(tr_ids) + len(va_ids)) / wall, 1) if wall > 0 else None,
        "from_cache": not need_gpu,
        "labels_used_in_pretraining": BACKBONE_SPECS[key]["labels_used"],
    })
    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\nfeatures listas para", list(FEATURES), "|", elapsed())

## 5. La rejilla de probes (aca sale el resultado)

Para cada backbone, cada probe, cada regimen de etiquetas y cada semilla:

1. Se toma el subconjunto de etiquetas del regimen (anidado: el 1 % esta contenido en el
   5 %, que esta contenido en el 10 %... asi la curva mide eficiencia de etiquetas y no
   ruido de muestreo).
2. Se ajusta el probe sobre las features de entrenamiento **ya extraidas**.
3. Se evalua en validacion: macro-F1, AUC, exactitud, ECE.

Tambien se cronometra todo. Esos tiempos alimentan el capitulo de hardware: permiten
comparar configuraciones **a igual presupuesto de computo** en vez de a igual numero de
epocas, que es la comparacion que la literatura casi nunca hace.

In [ ]:
rows = []
budget_hit = False
for key in BACKBONES:
    x_tr, y_tr = FEATURES[key]["train"]
    x_va, y_va = FEATURES[key]["val"]
    for seed in SEEDS:
        regimes = build_regimes(y_tr, FRACTIONS, seed=seed, min_per_class=2)
        for frac, idx in regimes.items():
            for probe in PROBES:
                if (time.time() - T_START) / 3600 > MAX_HOURS:
                    budget_hit = True
                    break
                t0 = time.time()
                try:
                    proba = run_probe(probe, x_tr[idx], y_tr[idx], x_va, N_CLASSES)
                    stats = summarize(y_va, proba, N_CLASSES)
                    error = None
                except Exception as exc:  # un probe caido no debe tumbar la rejilla
                    stats = {"macro_f1": float("nan"), "accuracy": float("nan"),
                             "auc": float("nan"), "ece": float("nan")}
                    error = f"{type(exc).__name__}: {exc}"
                    print(f"[WARN] {key}/{probe}/{frac:.0%}/s{seed} fallo -> {error}")
                rows.append({"backbone": key, "supervision": "ssl" if key.startswith("ssl_") else "supervised",
                             "probe": probe, "fraction": frac, "seed": seed,
                             "n_labels": int(len(idx)), "seconds": round(time.time() - t0, 3),
                             "error": error, **stats})
if budget_hit:
    print(f"[GUARDA DE TIEMPO] {elapsed()} paso de {MAX_HOURS} h -> rejilla truncada. "
          f"{len(rows)} combinaciones completadas (corte explicito, no silencioso).")

assert rows, ("la rejilla quedo vacia: ninguna combinacion corrio. "
              "Revisa MAX_HOURS, BACKBONES y PROBES antes de seguir.")
grid = pd.DataFrame(rows)
print(f"\n{len(grid)} combinaciones | fallos: {int(grid.error.notna().sum())} | {elapsed()}")

## 6. Resultados, tablas y el reporte de compuertas

Tres salidas:

1. **Tabla por regimen** - macro-F1 y AUC media +/- desviacion entre semillas. Es la tabla
   que va al informe.
2. **`RESULTS_SUMMARY_v5.json`** - todos los numeros crudos, para regenerar figuras sin
   volver a correr nada.
3. **Reporte de compuertas G0-G2** - criterios de aceptacion explicitos. Si una compuerta
   falla, se dice cual y por que; nada de "listo" sin evidencia.

In [ ]:
ok = grid[grid.error.isna()]
agg = (ok.groupby(["backbone", "supervision", "probe", "fraction"])
         .agg(macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
              auc_mean=("auc", "mean"), auc_std=("auc", "std"),
              n_labels=("n_labels", "max"), seeds=("seed", "nunique"))
         .reset_index().sort_values(["probe", "fraction", "auc_mean"], ascending=[True, True, False]))

pd.set_option("display.width", 170)
pd.set_option("display.max_rows", 300)
# std con 1 semilla es NaN a proposito: no se puede estimar dispersion de una muestra.
agg["metrica_titular"] = agg.probe.map(headline_metric)
print("=== macro-F1 y AUC por backbone / probe / regimen "
      "(std NaN = una sola semilla) ===")
print(agg.round(4).to_string(index=False))

print("\n=== ADVERTENCIA DE LECTURA ===")
print(f"  Probes dominados por el prior de clase: {PRIOR_DOMINATED_PROBES}")
print("  En esos probes el argmax sigue la clase mayoritaria (nv es ~65 %), asi que su")
print("  macro-F1 NO se cita como resultado: se lee su AUC, que es por ranking.")
print(f"  Probes sin calibracion real (su ECE no es una afirmacion): {UNCALIBRATED_PROBES}")

# --- delta SSL vs mejor supervisado, la comparacion titular -----------------
deltas = []
for probe in sorted(ok.probe.unique()):
    for frac in sorted(ok.fraction.unique()):
        sel = agg[(agg.probe == probe) & (agg.fraction == frac)]
        ssl_rows = sel[sel.supervision == "ssl"]
        sup_rows = sel[sel.supervision == "supervised"]
        if ssl_rows.empty or sup_rows.empty:
            continue
        metric = f"{headline_metric(probe)}_mean"
        best_ssl = ssl_rows.loc[ssl_rows[metric].idxmax()]
        best_sup = sup_rows.loc[sup_rows[metric].idxmax()]
        deltas.append({"probe": probe, "fraction": frac, "metrica": headline_metric(probe),
                       "mejor_ssl": best_ssl.backbone, "valor_ssl": round(best_ssl[metric], 4),
                       "mejor_sup": best_sup.backbone, "valor_sup": round(best_sup[metric], 4),
                       "delta": round(best_ssl[metric] - best_sup[metric], 4)})
delta_df = pd.DataFrame(deltas)
if len(delta_df):
    print("\n=== SSL vs mejor supervisado (en la metrica titular de cada probe) ===")
    print(delta_df.to_string(index=False))
    wins = int((delta_df.delta > 0).sum())
    print(f"\n  el SSL gana en {wins}/{len(delta_df)} combinaciones probe x regimen")

# --- compuertas de aceptacion (logica en dermssl.gates, cubierta por tests) --
gate_list = evaluate_gates(
    backend=data.backend,
    ham_images=len(data.ham_images),
    lesion_overlap=len(overlap),
    rows_total=len(grid),
    rows_ok=len(ok),
    frozen_verified=True,          # el assert de la celda de extraccion ya lo garantizo
    backbones=BACKBONES,
    grid_rows=json.loads(ok.to_json(orient="records")),
    smoke=SMOKE,
)

# --- equidad por fototipo (Fase 2: fst_manifest.csv, generado en Kaggle por --
# torch_unet.py, Fase 3 - dermssl.fitzpatrick). Si el manifiesto todavia no
# existe (por ejemplo, antes de correr la Fase 3), este bloque no toca
# gate_list: la corrida se comporta exactamente como v5 (design.md, tabla
# "Error Handling": "Manifest missing -> ... v5 behavior unchanged").
FST_MANIFEST_PATH = WORK / "fst_manifest.csv"
if FST_MANIFEST_PATH.is_file():
    fst_manifest = pd.read_csv(FST_MANIFEST_PATH)
    fst_by_id = fst_manifest.set_index("image_id")["fst_bucket"]
    ssl_key = next((k for k in BACKBONES if k.startswith("ssl_")), None)
    strat = {}
    if ssl_key is not None and "knn" in PROBES and 1.0 in FRACTIONS:
        x_tr, y_tr = FEATURES[ssl_key]["train"]
        x_va, y_va = FEATURES[ssl_key]["val"]
        proba_va = run_probe("knn", x_tr, y_tr, x_va, N_CLASSES)
        joined = [
            (label, proba, int(fst_by_id[img_id]))
            for img_id, label, proba in zip(va_ids, y_va, proba_va)
            if img_id in fst_by_id.index and not pd.isna(fst_by_id[img_id])
        ]
        if joined:
            labels_j, proba_j, buckets_j = zip(*joined)
            strat = stratified_summary(
                np.array(labels_j), np.array(proba_j), np.array(buckets_j), N_CLASSES
            )
    gate_list.append(fairness_gate(strat))

print()
print(report(gate_list))

summary = {
    "mode": "SMOKE (numeros NO validos)" if SMOKE else "corrida real",
    "notebook": "proyecto_ssl_lesiones_v5",
    "claim": "ninguna etiqueta toca el extractor de features; backbone congelado",
    "elapsed": elapsed(),
    "data": {"backend": data.backend, "ham_images": len(data.ham_images),
             "pool_images": data.summary()["pool_images"], "train": len(tr_ids),
             "val": len(va_ids), "classes": classes, "lesion_overlap": int(len(overlap))},
    "backbones": {k: {**BACKBONE_SPECS[k], "resolution": RES_BY_BACKBONE[k]} for k in BACKBONES},
    "reporting_rules": {"prior_dominated_probes": list(PRIOR_DOMINATED_PROBES),
                        "uncalibrated_probes": list(UNCALIBRATED_PROBES)},
    "extraction": EXTRACTION_LOG,
    "grid": json.loads(grid.to_json(orient="records")),
    "aggregate": json.loads(agg.to_json(orient="records")),
    "ssl_vs_supervised": deltas,
    "gates": [g.as_dict() for g in gate_list],
    "all_gates_pass": bool(all_blocking_pass(gate_list)),
}
out_path = WORK / "RESULTS_SUMMARY_v5.json"
out_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
grid.to_csv(WORK / "probe_grid_v5.csv", index=False)
print(f"\nguardado -> {out_path} y probe_grid_v5.csv | {elapsed()}")